# Online Retail: Customer Analytics & Segmentation
**Capstone project** · Dataset: UCI *Online Retail* (UK-based online gift retailer, 1 Dec 2010 - 9 Dec 2011)

**How to run:** put `Online Retail.xlsx` in `data/raw/`, then *Restart & Run All*. The notebook is deterministic (`SEED = 42`) and writes the tables used by the Streamlit app to `data/processed/`.

| # | Section | # | Section |
|---|---|---|---|
| 1 | Business Understanding | 9 | RFM Analysis |
| 2 | Dataset Loading & Documentation | 10 | RFM Visualization & Interpretation |
| 3 | Data Audit | 11 | Preprocessing for Clustering |
| 4 | Cleaning & Data Quality Decisions | 12 | Clustering & Evaluation |
| 5 | Feature Engineering | 13 | Segment Profiling |
| 6 | EDA | 14 | Business Recommendations |
| 7 | Business Insights | 15 | Limitations & Next Steps |
| 8 | Customer-Level Feature Engineering | 16 | Final Conclusion |

## 1. Business Understanding

**Business goal:** understand how the business earns its money and turn 541k transaction lines into customer groups that each call for a different marketing action.

**Questions this notebook answers**
1. Which products, customers, countries and time periods drive revenue?
2. How common are cancellations and returns, and where do they come from?
3. Is there bulk / wholesale-like buying behaviour?
4. Can customers be grouped into segments based on value and engagement, and what should we do with each segment?

**Approach:** audit and clean the data with every decision logged (business impact included) -> EDA before any modelling -> RFM -> clustering (K-Means with Hierarchical and DBSCAN as checks) -> segment profiles backed by statistics -> actions.

**Setup cell:** imports, the random seed, paths and two registers (`decision_log`, `insights`) that fill up as we go.

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
from IPython.display import display

from scipy.cluster.hierarchy import linkage
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)

SEED = 42  # fixed seed so the results are reproducible

# Works whether the notebook is run from the repo root or from notebooks/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_FILE = ROOT / "data" / "raw" / "Online Retail.xlsx"
OUT = ROOT / "data" / "processed"

# Two registers filled as we go: every cleaning decision and every business insight
decision_log = []
insights = []

def log_decision(step, decision, rows_affected, business_impact):
    decision_log.append({"step": step, "decision": decision,
                         "rows_affected": rows_affected, "business_impact": business_impact})

def add_insight(text):
    insights.append(text)


## 2. Dataset Loading & Documentation

**Business intent:** know exactly what one row means before computing anything. **One row is one transaction line (one product on one invoice), not an order.**

| Column | Meaning |
|---|---|
| `InvoiceNo` | Invoice id. Starts with **C** = cancellation / return (negative quantity) |
| `StockCode` | Product code (a few are non-products: postage, fees, manual adjustments) |
| `Description` | Product name |
| `Quantity` | Units on the line (negative on cancellations and stock adjustments) |
| `InvoiceDate` | Timestamp of the invoice |
| `UnitPrice` | Price per unit (GBP) |
| `CustomerID` | Customer id (missing for guest checkouts). **A label, never a numeric feature** |
| `Country` | Customer country |

IDs and codes are read as text on purpose, so `85123A` and `0123` keep their form.

In [2]:
df = pd.read_excel(
    RAW_FILE,
    dtype={"InvoiceNo": str, "StockCode": str, "Description": str},
)

print("Shape:", df.shape)
df.head()

Shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


In [4]:
df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
InvoiceNo,541909,25900,573585,1114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,541909,4070,85123A,2313,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,540455,4223,WHITE HANGING HEART T-LIGHT HOLDER,2369,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,541909.0,NaN,NaN,NaN,9.55225,-80995.0,1.0,3.0,10.0,80995.0,218.081158
InvoiceDate,541909,NaN,NaN,NaN,2011-07-04 13:34:57.156386048,2010-12-01 08:26:00,2011-03-28 11:34:00,2011-07-19 17:17:00,2011-10-19 11:27:00,2011-12-09 12:50:00,NaN
UnitPrice,541909.0,NaN,NaN,NaN,4.611114,-11062.06,1.25,2.08,4.13,38970.0,96.759853
CustomerID,406829.0,NaN,NaN,NaN,15287.69057,12346.0,13953.0,15152.0,16791.0,18287.0,1713.600303
Country,541909,38,United Kingdom,495478,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Data Audit

**Business intent:** find out what is wrong with the data *before* deciding what to do about it. We audit three things the guideline singles out: missing `CustomerID` (are they guest-checkout invoices?), cancellations, and negative / zero-quantity rows that are not cancellations. Nothing is dropped in this section.

In [5]:
# Keep the original untouched; all cleaning happens on df
df_raw = df.copy()
print("Rows:", len(df))

Rows: 541909


In [6]:
# Per invoice: how many rows, and how many of them are missing CustomerID?
inv_check = df.groupby("InvoiceNo")["CustomerID"].agg(
    rows="size",
    missing=lambda s: s.isna().sum(),
)

inv_check["kind"] = np.select(
    [inv_check["missing"] == 0, inv_check["missing"] == inv_check["rows"]],
    ["all rows have ID", "all rows missing ID"],
    default="mixed (some rows missing)",
)

print(inv_check["kind"].value_counts())
print("\nShare of all rows missing CustomerID:", round(df["CustomerID"].isna().mean() * 100, 2), "%")

kind
all rows have ID       22190
all rows missing ID     3710
Name: count, dtype: int64

Share of all rows missing CustomerID: 24.93 %


In [7]:
df["IsCancellation"] = df["InvoiceNo"].str.startswith("C")
neg_qty = df["Quantity"] < 0

print("Cancellation rows:", df["IsCancellation"].sum(),
      "| invoices:", df.loc[df["IsCancellation"], "InvoiceNo"].nunique())
print("Share of invoices that are cancellations:",
      round(df.loc[df["IsCancellation"], "InvoiceNo"].nunique() / df["InvoiceNo"].nunique() * 100, 2), "%")
print("Cancellation rows with Quantity >= 0:", (df["IsCancellation"] & ~neg_qty).sum())

# Negative quantity but NOT a 'C' invoice: what are these?
odd_neg = df[neg_qty & ~df["IsCancellation"]]
print("\nNegative qty but NOT a 'C' invoice:", len(odd_neg))
print("  with missing CustomerID:", odd_neg["CustomerID"].isna().sum())
print("  with UnitPrice == 0:", (odd_neg["UnitPrice"] == 0).sum())
odd_neg[["InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "CustomerID"]].head(10)

Cancellation rows: 9288 | invoices: 3836
Share of invoices that are cancellations: 14.81 %
Cancellation rows with Quantity >= 0: 0

Negative qty but NOT a 'C' invoice: 1336
  with missing CustomerID: 1336
  with UnitPrice == 0: 1336


,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID
2406,536589,21777,NaN,-10,0.0,NaN
4347,536764,84952C,NaN,-38,0.0,NaN
7188,536996,22712,NaN,-20,0.0,NaN
7189,536997,22028,NaN,-20,0.0,NaN
7190,536998,85067,NaN,-6,0.0,NaN
7192,537000,21414,NaN,-22,0.0,NaN
7193,537001,21653,NaN,-6,0.0,NaN
7195,537003,85126,NaN,-2,0.0,NaN
7196,537004,21814,NaN,-30,0.0,NaN
7197,537005,21692,NaN,-70,0.0,NaN


## 4. Cleaning & Data Quality Decisions

**Business intent:** clean data guarantees reliable business logic. The rule used throughout: **flag first, drop only when there is a reason, and log every decision with its business impact** (`decision_log`).

Steps: type casting and time features -> `CustomerID` as text with a `HasCustomerID` flag (not dropped) -> stock adjustments / zero-price / negative-price flags -> match cancellations to their original purchases.

In [8]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

df["Year"]         = df["InvoiceDate"].dt.year
df["Month"]        = df["InvoiceDate"].dt.month
df["DayOfWeek"]    = df["InvoiceDate"].dt.day_name()
df["DayOfWeekNum"] = df["InvoiceDate"].dt.dayofweek   # 0 = Monday, to sort days correctly in charts
df["Hour"]         = df["InvoiceDate"].dt.hour
df["YearMonth"]    = df["InvoiceDate"].dt.to_period("M").astype(str)  # e.g. 2011-03, needed for the monthly revenue trend

print(df["InvoiceDate"].dtype)
print("Date range:", df["InvoiceDate"].min(), "->", df["InvoiceDate"].max())
df[["InvoiceDate", "Year", "Month", "DayOfWeek", "Hour", "YearMonth"]].head()
log_decision("Types", "InvoiceDate -> datetime; CustomerID float -> text ID; ID/code columns read as text",
             len(df), "IDs are labels (never used as numeric features); dates enable time analysis")
log_decision("Time features", "Added Year, Month, DayOfWeek, DayOfWeekNum, Hour, YearMonth",
             len(df), "Enables seasonality, weekday and hourly analysis. Note: Dec 2011 is incomplete (data ends on the 9th)")

datetime64[ns]
Date range: 2010-12-01 08:26:00 -> 2011-12-09 12:50:00


In [9]:
# CustomerID is an identifier, not a number: float -> nullable int -> text
df["CustomerID"] = df["CustomerID"].astype("Int64").astype("string")

# Flag instead of dropping
df["HasCustomerID"] = df["CustomerID"].notna()

# How big is this group really? Compare rows, invoices and revenue share
rev = df["Quantity"] * df["UnitPrice"]
size_check = pd.DataFrame({
    "rows_%": df.groupby("HasCustomerID").size() / len(df) * 100,
    "invoices": df.groupby("HasCustomerID")["InvoiceNo"].nunique(),
    "net_revenue_%": rev.groupby(df["HasCustomerID"]).sum() / rev.sum() * 100,
}).round(2)
display(size_check)

log_decision(
    "Missing CustomerID",
    "Kept in df and flagged (HasCustomerID). Used for sales/product/time/country EDA; excluded from RFM and clustering",
    int((~df["HasCustomerID"]).sum()),
    "Revenue totals stay complete; customer-level analysis only uses identifiable customers",
)

,rows_%,invoices,net_revenue_%
HasCustomerID,,,
False,24.93,3710,14.85
True,75.07,22190,85.15


In [10]:
# Flag instead of drop: these rows are not customer sales
df["IsStockAdjustment"] = (df["Quantity"] < 0) & ~df["IsCancellation"] & (df["UnitPrice"] == 0)
df["IsZeroPrice"]       = (df["UnitPrice"] == 0) & ~df["IsStockAdjustment"]   # free items / giveaways
df["IsNegativePrice"]   = df["UnitPrice"] < 0                                   # accounting adjustments

print(df[["IsStockAdjustment", "IsZeroPrice", "IsNegativePrice"]].sum())

log_decision("Stock adjustments", "Flagged (neg qty, non-C invoice, price 0, no ID); excluded from revenue and customer analysis",
             int(df["IsStockAdjustment"].sum()), "Not customer sales or returns; keeping them would distort revenue and basket sizes")
log_decision("Zero price", "Flagged (price 0, not an adjustment); excluded from revenue and RFM",
             int(df["IsZeroPrice"].sum()), "Free items add no revenue but would inflate order counts and quantities")
log_decision("Negative price", "Flagged; excluded from analysis",
             int(df["IsNegativePrice"].sum()), "Accounting adjustments, not product sales")

IsStockAdjustment    1336
IsZeroPrice          1179
IsNegativePrice         2
dtype: int64


In [11]:
df[df["IsNegativePrice"]]

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,IsCancellation,Year,Month,DayOfWeek,DayOfWeekNum,Hour,YearMonth,HasCustomerID,IsStockAdjustment,IsZeroPrice,IsNegativePrice
299983,A563186,B,Adjust bad debt,1,2011-08-12 14:51:00,-11062.06,<NA>,United Kingdom,False,2011,8,Friday,4,14,2011-08,False,False,False,True
299984,A563187,B,Adjust bad debt,1,2011-08-12 14:52:00,-11062.06,<NA>,United Kingdom,False,2011,8,Friday,4,14,2011-08,False,False,False,True


In [12]:
# Valid purchases we can match against (real sales to identifiable customers)
purch = df[(df["Quantity"] > 0) & ~df["IsZeroPrice"] & ~df["IsNegativePrice"] & df["HasCustomerID"]]

canc = df[df["IsCancellation"] & df["HasCustomerID"]].copy()
canc["ReturnQty"] = -canc["Quantity"]
canc = canc.reset_index().rename(columns={"index": "cancel_idx"})

# All candidate purchases for the same customer + product
m = canc[["cancel_idx", "CustomerID", "StockCode", "InvoiceDate", "ReturnQty"]].merge(
    purch[["CustomerID", "StockCode", "InvoiceDate", "Quantity"]],
    on=["CustomerID", "StockCode"], how="left", suffixes=("_c", "_p"),
)
m["prior"] = m["InvoiceDate_p"] <= m["InvoiceDate_c"]            # bought before the cancellation
m["full"]  = m["prior"] & (m["Quantity"] >= m["ReturnQty"])      # and bought enough units

res = m.groupby("cancel_idx").agg(any_prior=("prior", "max"), any_full=("full", "max"))
canc = canc.merge(res, left_on="cancel_idx", right_index=True, how="left")

canc["MatchStatus"] = np.select(
    [canc["any_full"] == True, canc["any_prior"] == True],
    ["matched (full qty)", "matched (partial qty)"],
    default="no original found",
)

canc["Revenue"] = canc["Quantity"] * canc["UnitPrice"]   # negative
summary = canc.groupby("MatchStatus").agg(
    rows=("MatchStatus", "size"),
    refunded_value=("Revenue", "sum"),
)
summary["rows_%"] = (summary["rows"] / summary["rows"].sum() * 100).round(1)
display(summary)

print("Cancellation rows without CustomerID (cannot be matched):", (df["IsCancellation"] & ~df["HasCustomerID"]).sum())

,rows,refunded_value,rows_%
MatchStatus,,,
matched (full qty),7511,-475889.30,84.3
matched (partial qty),88,-11218.81,1.0
no original found,1306,-124233.98,14.7


Cancellation rows without CustomerID (cannot be matched): 383


In [13]:
# Bring the match result back to df
df["MatchStatus"] = "not a cancellation"
df.loc[df["IsCancellation"] & ~df["HasCustomerID"], "MatchStatus"] = "cancellation, no CustomerID"
df.loc[canc["cancel_idx"], "MatchStatus"] = canc["MatchStatus"].values

print(df["MatchStatus"].value_counts())

log_decision(
    "Cancellations",
    "Kept in df (negative revenue nets out automatically); flagged IsCancellation + MatchStatus (matched / partial / no original / no CustomerID)",
    int(df["IsCancellation"].sum()),
    "Net revenue is correct; return analysis can separate verified returns from unmatched ones (14.7% unmatched, may refer to pre-2010-12 purchases)",
)

MatchStatus
not a cancellation             532621
matched (full qty)               7511
no original found                1306
cancellation, no CustomerID       383
matched (partial qty)              88
Name: count, dtype: int64


## 5. Feature Engineering

**Business intent:** create the two features the whole analysis relies on. `TotalLineRevenue = Quantity * UnitPrice` (negative on returns, so revenue nets out automatically) and `Is_UK` to separate international behaviour from the UK majority.

In [14]:
df["TotalLineRevenue"] = df["Quantity"] * df["UnitPrice"]
df["Is_UK"] = df["Country"] == "United Kingdom"

print("UK share of rows:", round(df["Is_UK"].mean() * 100, 1), "%")
print("UK share of net revenue:", round(df.loc[df["Is_UK"], "TotalLineRevenue"].sum() / df["TotalLineRevenue"].sum() * 100, 1), "%")

gross   = df.loc[df["TotalLineRevenue"] > 0, "TotalLineRevenue"].sum()
returns = df.loc[df["IsCancellation"], "TotalLineRevenue"].sum()
print(f"\nGross sales: {gross:,.0f} | Cancellations: {returns:,.0f} | Net (all rows): {df['TotalLineRevenue'].sum():,.0f}")

UK share of rows: 91.4 %
UK share of net revenue: 84.0 %

Gross sales: 10,666,685 | Cancellations: -896,812 | Net (all rows): 9,747,748


### 5.2 Remaining cleaning that needs the revenue column: non-product codes and duplicates

**Business intent:** postage, fees and manual adjustments are not products, and exact duplicate rows would double-count units. Both are handled here because deciding on them needs `TotalLineRevenue`. The result is `sales`: real product lines (cancellations stay in, so returns still net out).

In [15]:
# Non-product StockCodes: what are they and how much revenue do they carry?
non_prod = df[~df["StockCode"].str.fullmatch(r"\d{5}[A-Za-z]{0,2}", na=False)]
display(
    non_prod.groupby("StockCode").agg(
        rows=("StockCode", "size"),
        description=("Description", lambda s: s.dropna().iloc[0] if s.notna().any() else None),
        revenue=("TotalLineRevenue", "sum"),
    ).sort_values("rows", ascending=False).head(15)
)

print("Fully duplicated rows:", df.duplicated().sum())

,rows,description,revenue
StockCode,,,
POST,1256,POSTAGE,66230.640
DOT,710,DOTCOM POSTAGE,206245.480
M,571,Manual,-68674.190
C2,144,CARRIAGE,6986.000
D,77,Discount,-5696.220
S,63,SAMPLES,-3049.390
BANK CHARGES,37,Bank Charges,-7175.639
AMAZONFEE,34,AMAZON FEE,-221520.500
CRUK,16,CRUK Commission,-7933.430


Fully duplicated rows: 5268


In [16]:
NON_PRODUCT_CODES = ["POST", "DOT", "M", "C2", "D", "S", "BANK CHARGES", "AMAZONFEE", "CRUK", "B"]
df["IsNonProduct"] = df["StockCode"].isin(NON_PRODUCT_CODES)

print("Non-product rows:", df["IsNonProduct"].sum(),
      "| revenue:", round(df.loc[df["IsNonProduct"], "TotalLineRevenue"].sum()))

# Any other odd codes we have not classified yet?
rest = non_prod[~non_prod["StockCode"].isin(NON_PRODUCT_CODES)]
print("\nOther non-standard codes:")
print(rest["StockCode"].value_counts().head(10))

log_decision("Non-product codes", "Flagged IsNonProduct (postage, fees, manual, discount, samples); excluded from product and basket analysis",
             int(df["IsNonProduct"].sum()), "Fees and postage are not products; keeping them would pollute top-product rankings and basket sizes")

Non-product rows: 2911 | revenue: -45649

Other non-standard codes:
StockCode
DCGSSGIRL       13
DCGSSBOY        11
gift_0001_20    10
gift_0001_10     9
gift_0001_30     8
DCGS0003         5
gift_0001_50     4
PADS             4
gift_0001_40     3
DCGS0076         2
Name: count, dtype: int64


In [17]:
check_codes = ["DCGSSGIRL", "DCGSSBOY", "DCGS0003", "PADS", "B"]
display(df[df["StockCode"].isin(check_codes) | df["StockCode"].str.startswith("gift_", na=False)]
        .groupby("StockCode").agg(rows=("StockCode", "size"),
                                   description=("Description", lambda s: s.dropna().iloc[0] if s.notna().any() else None),
                                   avg_price=("UnitPrice", "mean")))
log_decision("Gift vouchers / odd codes", "Kept as normal sales: gift_0001_* vouchers, DCGS party bags/ashtray, PADS (near-zero price) reviewed individually",
             int(df["StockCode"].str.startswith("gift_", na=False).sum()),
             "Small volume (about 40 rows); real revenue, so kept; vouchers are not physical demand, noted as a limitation")

,rows,description,avg_price
StockCode,,,
B,3,Adjust bad debt,-3687.353333
DCGS0003,5,BOXED GLASS ASHTRAY,1.988000
DCGSSBOY,11,BOYS PARTY BAG,3.104545
DCGSSGIRL,13,GIRLS PARTY BAG,3.133077
PADS,4,PADS TO MATCH ALL CUSHIONS,0.000750
gift_0001_10,9,Dotcomgiftshop Gift Voucher £10.00,7.404444
gift_0001_20,10,Dotcomgiftshop Gift Voucher £20.00,15.038000
gift_0001_30,8,Dotcomgiftshop Gift Voucher £30.00,21.941250
gift_0001_40,3,Dotcomgiftshop Gift Voucher £40.00,33.566667


In [18]:
base_cols = list(df_raw.columns)   # the 8 original columns

full_dup = df.duplicated(subset=base_cols)
same_except_invoice = df.duplicated(subset=[c for c in base_cols if c != "InvoiceNo"]) & ~full_dup

print("Identical in ALL columns (incl. InvoiceNo):", full_dup.sum())
print("Identical except InvoiceNo:", same_except_invoice.sum())

dups = df[df.duplicated(subset=base_cols, keep=False)]
print("Invoices containing duplicates:", dups["InvoiceNo"].nunique())
print("Revenue carried by the extra copies:", round(df.loc[full_dup, "TotalLineRevenue"].sum()))
dups.sort_values(base_cols).head(10)[["InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate", "UnitPrice", "CustomerID"]]

Identical in ALL columns (incl. InvoiceNo): 5268
Identical except InvoiceNo: 580


Invoices containing duplicates: 1933
Revenue carried by the extra copies: 21741


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920
598,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920


In [19]:
n_before = len(df)
df = df.drop_duplicates(subset=base_cols, keep="first").reset_index(drop=True)

log_decision("Duplicates", "Dropped rows identical in all 8 original columns (kept first copy)",
             n_before - len(df),
             "Removes ~0.2% of revenue; prevents double-counted units and inflated basket sizes. Caveat: a genuine repeated scan of the same item would also be removed")
print("Rows:", n_before, "->", len(df))

Rows: 541909 -> 536641


In [20]:
# Product-level sales view: real product lines only (cancellations stay in, so revenue nets out)
sales = df[~(df["IsStockAdjustment"] | df["IsZeroPrice"] | df["IsNegativePrice"] | df["IsNonProduct"])].copy()
print("df:", len(df), "| sales (product lines):", len(sales))

df: 536641 | sales (product lines): 531240


In [21]:
# Decision log so far (it keeps growing in later sections)
pd.DataFrame(decision_log)

,step,decision,rows_affected,business_impact
0,Types,InvoiceDate -> datetime; CustomerID float -> text ID; ID/code columns read as text,541909,IDs are labels (never used as numeric features); dates enable time analysis
1,Time features,"Added Year, Month, DayOfWeek, DayOfWeekNum, Hour, YearMonth",541909,"Enables seasonality, weekday and hourly analysis. Note: Dec 2011 is incomplete (data ends on the 9th)"
2,Missing CustomerID,Kept in df and flagged (HasCustomerID). Used for sales/product/time/country EDA; excluded from RFM and clustering,135080,Revenue totals stay complete; customer-level analysis only uses identifiable customers
3,Stock adjustments,"Flagged (neg qty, non-C invoice, price 0, no ID); excluded from revenue and customer analysis",1336,Not customer sales or returns; keeping them would distort revenue and basket sizes
4,Zero price,"Flagged (price 0, not an adjustment); excluded from revenue and RFM",1179,Free items add no revenue but would inflate order counts and quantities
5,Negative price,Flagged; excluded from analysis,2,"Accounting adjustments, not product sales"
6,Cancellations,Kept in df (negative revenue nets out automatically); flagged IsCancellation + MatchStatus (matched / partial / no o...,9288,"Net revenue is correct; return analysis can separate verified returns from unmatched ones (14.7% unmatched, may refe..."
7,Non-product codes,"Flagged IsNonProduct (postage, fees, manual, discount, samples); excluded from product and basket analysis",2911,Fees and postage are not products; keeping them would pollute top-product rankings and basket sizes
8,Gift vouchers / odd codes,"Kept as normal sales: gift_0001_* vouchers, DCGS party bags/ashtray, PADS (near-zero price) reviewed individually",34,"Small volume (about 40 rows); real revenue, so kept; vouchers are not physical demand, noted as a limitation"
9,Duplicates,Dropped rows identical in all 8 original columns (kept first copy),5268,Removes ~0.2% of revenue; prevents double-counted units and inflated basket sizes. Caveat: a genuine repeated scan o...


## 6. Exploratory Data Analysis

**Business intent:** decide what the business needs to understand *before* any clustering. Each sub-section answers one executive question and ends with an interpretation. Unless stated otherwise, EDA uses `sales` (product lines only). Customer-level views use only identifiable customers.

### 6.1 Products: which generate the most units and the most revenue?
**Business intent:** separate products that *move* from products that *earn*.

In [22]:
# Products: units vs revenue
name_of = sales.dropna(subset=["Description"]).groupby("StockCode")["Description"].agg(lambda s: s.mode().iloc[0])

prod = sales.groupby("StockCode").agg(
    units=("Quantity", "sum"),
    revenue=("TotalLineRevenue", "sum"),
    n_invoices=("InvoiceNo", "nunique"),
)
prod["name"] = name_of

top_units = prod.nlargest(10, "units").iloc[::-1]
top_rev   = prod.nlargest(10, "revenue").iloc[::-1]

fig = px.bar(top_units, x="units", y="name", orientation="h", hover_data=["revenue", "n_invoices"],
             title="Top 10 products by net units sold", color_discrete_sequence=["#08D9D6"])
fig.show()

fig = px.bar(top_rev, x="revenue", y="name", orientation="h", hover_data=["units", "n_invoices"],
             title="Top 10 products by net revenue", color_discrete_sequence=["#FF2E63"])
fig.show()

both = set(top_units.index) & set(top_rev.index)
print("Products in BOTH top-10 lists:", len(both))

Products in BOTH top-10 lists: 5


In [23]:
# Is high volume driven by many orders or a few bulk orders? Compare the top lists side by side.
check = prod.loc[list(set(top_units.index) | set(top_rev.index))].copy()
check["avg_price"] = (check["revenue"] / check["units"]).round(2)
check["units_per_invoice"] = (check["units"] / check["n_invoices"]).round(1)
check.sort_values("revenue", ascending=False)[["name", "units", "revenue", "n_invoices", "avg_price", "units_per_invoice"]]

,name,units,revenue,n_invoices,avg_price,units_per_invoice
StockCode,,,,,,
22423,REGENCY CAKESTAND 3 TIER,12996,164459.49,2168,12.65,6.0
47566,PARTY BUNTING,18006,98243.88,1705,5.46,10.6
85123A,WHITE HANGING HEART T-LIGHT HOLDER,35063,97838.45,2240,2.79,15.7
85099B,JUMBO BAG RED RETROSPOT,47256,92175.79,2132,1.95,22.2
23084,RABBIT NIGHT LIGHT,30631,66661.63,1009,2.18,30.4
22086,PAPER CHAIN KIT 50'S CHRISTMAS,18876,63715.24,1170,3.38,16.1
84879,ASSORTED COLOUR BIRD ORNAMENT,36282,58792.42,1467,1.62,24.7
79321,CHILLI LIGHTS,10222,53746.66,667,5.26,15.3
22502,PICNIC BASKET WICKER SMALL,1862,51023.52,472,27.40,3.9


**Interpretation.** Units and revenue tell different stories: only 5 products are in both top-10 lists. The units leaders are cheap lines (Popcorn Holder: 56,427 units but ~0.90 per unit, so 51k revenue), while the revenue leaders are mid-priced or pricier (Regency Cakestand 3 Tier: 164k from 12,996 units at ~12.65; Picnic Basket Wicker Small: only 1,862 units but 51k at ~27). Cheap items are also bought in much larger quantities per invoice (World War 2 Gliders ~100 units per invoice, Popcorn Holder ~39, Regency Cakestand 6).

**So what:** rank and promote products by revenue, not units. A units ranking mostly shows low-price lines that suit bulk buyers.

### 6.2 Customers: who contributes the most revenue and orders?
**Business intent:** measure how dependent the business is on a few customers.

In [24]:
# Customers: who drives revenue and orders? (identifiable customers only)
cust = sales[sales["HasCustomerID"]].groupby("CustomerID").agg(
    revenue=("TotalLineRevenue", "sum"),
    orders=("InvoiceNo", "nunique"),
)

top_c = cust.nlargest(10, "revenue").reset_index().iloc[::-1]
fig = px.bar(top_c, x="revenue", y="CustomerID", orientation="h", hover_data=["orders"],
             title="Top 10 customers by net revenue", color_discrete_sequence=["#FF2E63"])
fig.update_yaxes(type="category")
fig.show()

total = cust["revenue"].sum()
s = cust["revenue"].sort_values(ascending=False)
print(f"Customers: {len(cust)}")
print(f"Top 10 customers share of identified revenue: {s.head(10).sum() / total:.1%}")
print(f"Top 20% of customers share of identified revenue: {s.head(int(len(s) * 0.2)).sum() / total:.1%}")
print(f"Customers with 1 order only: {(cust['orders'] == 1).mean():.1%}")

Customers: 4362
Top 10 customers share of identified revenue: 16.6%
Top 20% of customers share of identified revenue: 74.0%
Customers with 1 order only: 30.4%


**Interpretation.** Revenue depends on a small group: the top 10 customers generate 16.6% of identified revenue and the top 20% of customers generate 74.0%. The largest customer (14646) alone is about 279k. At the other end, 30.4% of customers bought only once in the period.

*Reading note:* "orders" in this section counts every invoice number, including cancellation invoices, and it covers 4,362 customers. Section 8 counts only valid purchase invoices and has 4,333 customers (29 customers only have cancellation lines). That is why a few numbers differ slightly between the two sections (e.g. customer 14911: 242 invoices here, 196 purchases in section 8).

### 6.3 Revenue over time
**Business intent:** spot the Q4 holiday surge and check data completeness at the edges of the period.

In [25]:
# Revenue over time (net: cancellations are inside, so returns reduce the month they happened in)
monthly = sales.groupby("YearMonth").agg(
    revenue=("TotalLineRevenue", "sum"),
    orders=("InvoiceNo", lambda s: s[~s.str.startswith("C")].nunique()),
).reset_index()
monthly["partial_month"] = monthly["YearMonth"] == monthly["YearMonth"].max()   # data stops on 9 Dec 2011

fig = px.bar(monthly, x="YearMonth", y="revenue", color="partial_month", hover_data=["orders"],
             color_discrete_map={False: "#08D9D6", True: "#FF2E63"},
             title="Monthly net revenue (red = incomplete month)")
fig.update_xaxes(type="category")
fig.show()

display(monthly.sort_values("revenue", ascending=False).head(5))

,YearMonth,revenue,orders,partial_month
11,2011-11,1427124.540,2751,False
10,2011-10,1061717.890,2005,False
9,2011-09,1011364.741,1818,False
0,2010-12,758169.640,1550,False
5,2011-05,731088.100,1668,False


**Interpretation.** From Jan to Aug 2011 monthly revenue is fairly flat (482k to 731k, average ~634k). It then climbs: Sep 1.01M, Oct 1.06M and **Nov 1.43M (the peak, ~2.3x the Jan-Aug average)**. Sep-Nov is 36% of the whole period. Dec 2010 (758k) is also one of the five strongest months, which points the same way as the Q4 pattern. December 2011 (red) has only 9 days of data, so its 440k is **not** a decline.

**So what:** this is consistent with a Q4 holiday surge, but with a single year it is an observation, not a confirmed yearly pattern. Plan Q4 stock early as a hypothesis and validate with a second year.

### 6.4 Weekdays and hours
**Business intent:** know when customers buy, to plan ad spend and server maintenance. Only purchases are used (a cancellation timestamp is not a purchase time).

In [26]:
# Weekday and hour patterns: purchases only (cancellation timestamps are not purchase times)
buys = sales[~sales["IsCancellation"]]

dow = buys.groupby(["DayOfWeekNum", "DayOfWeek"]).agg(
    revenue=("TotalLineRevenue", "sum"),
    orders=("InvoiceNo", "nunique"),
).reset_index()
fig = px.bar(dow, x="DayOfWeek", y="revenue", hover_data=["orders"],
             title="Revenue by weekday", color_discrete_sequence=["#08D9D6"])
fig.show()

hour = buys.groupby("Hour").agg(
    revenue=("TotalLineRevenue", "sum"),
    orders=("InvoiceNo", "nunique"),
).reset_index()
fig = px.bar(hour, x="Hour", y="revenue", hover_data=["orders"],
             title="Revenue by hour of day", color_discrete_sequence=["#FF2E63"])
fig.show()

print("Weekdays present in the data:", sorted(dow["DayOfWeek"].tolist()))

Weekdays present in the data: ['Friday', 'Monday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']


In [27]:
# Insights from 6.1-6.4 (every number is computed, none typed by hand)
u_lead, r_lead = top_units.iloc[-1], top_rev.iloc[-1]
u_rank_in_rev = int(prod["revenue"].rank(ascending=False)[top_units.index[-1]])
r_rank_in_units = int(prod["units"].rank(ascending=False)[top_rev.index[-1]])
add_insight(f"Products: only {len(both)} of the top-10 products by units are also in the top-10 by revenue. "
            f"{u_lead['name'].title()} leads in units (~{u_lead['units']/1000:.0f}k) but ranks {u_rank_in_rev} by revenue; "
            f"{r_lead['name'].title()} leads revenue (~{r_lead['revenue']/1000:.0f}k) but ranks {r_rank_in_units} by units. "
            f"Price per unit, not just volume, decides what drives revenue.")

tot_rev = monthly["revenue"].sum()
q4 = monthly[monthly["YearMonth"].isin(["2011-09", "2011-10", "2011-11"])]["revenue"].sum()
jan_aug = monthly[monthly["YearMonth"].between("2011-01", "2011-08")]["revenue"].mean()
nov = monthly.loc[monthly["YearMonth"] == "2011-11", "revenue"].iloc[0]
add_insight(f"Time: revenue from Sep-Nov 2011 is ~{q4/1e6:.1f}M, about {q4/tot_rev:.0%} of the whole period, and Nov 2011 ({nov/1e6:.2f}M) is "
            f"~{nov/jan_aug:.1f}x the Jan-Aug monthly average (~{jan_aug/1e3:.0f}k). "
            f"Descriptive only: with ~1 year of data we cannot call it a recurring seasonal pattern, and Dec 2011 is incomplete (ends on the 9th).")

add_insight(f"Customers: the top 20% of identifiable customers generate {s.head(int(len(s) * 0.2)).sum() / total:.0%} of identified revenue, "
            f"and {(cust['orders'] == 1).mean():.1%} of customers bought only once. "
            f"Revenue is concentrated, and a large share of customers never came back in the observed period.")

top_hours = hour.nlargest(5, "revenue")["Hour"].tolist()
no_days = [d for d in ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"] if d not in set(dow["DayOfWeek"])]
add_insight(f"Hours/days: revenue peaks in the hours {min(top_hours)}:00-{max(top_hours)}:59 (top hours: {', '.join(map(str, top_hours))}) "
            f"and no transactions are recorded on {', '.join(no_days)}.")
print(len(insights), "insights so far")

4 insights so far


**Interpretation.** This is a weekday, daytime business. 74.4% of purchase revenue falls between 10:00 and 15:59, with 12:00 (1.41M) and 10:00 (1.40M) the strongest hours, and only 6.3% comes after 17:00. Thursday (2.13M, 20.8%) and Tuesday (2.09M, 20.3%) are the strongest days; Sunday is the weakest (7.8%). **No transactions are recorded on Saturdays.**

**So what:** test ad timing and email sends in weekday mid-morning to mid-afternoon; schedule server maintenance in the evening or early morning, when volume is lowest. Whether Saturday is really closed or just not recorded cannot be answered from this data, so confirm before relying on it.

### 6.5 Geography: customers, orders and revenue by country
**Business intent:** isolate international trends from the UK skew.

In [28]:
# Countries: customers, orders and revenue
country = sales.groupby("Country").agg(
    revenue=("TotalLineRevenue", "sum"),
    orders=("InvoiceNo", lambda s: s[~s.str.startswith("C")].nunique()),
    customers=("CustomerID", "nunique"),
).sort_values("revenue", ascending=False)
country["revenue_share_%"] = (country["revenue"] / country["revenue"].sum() * 100).round(1)
country["avg_order_value"] = (country["revenue"] / country["orders"]).round(0)

display(country.head(10))

non_uk = country.drop("United Kingdom").head(10).reset_index().iloc[::-1]
fig = px.bar(non_uk, x="revenue", y="Country", orientation="h", hover_data=["orders", "customers", "avg_order_value"],
             title="Top 10 non-UK countries by net revenue", color_discrete_sequence=["#08D9D6"])
fig.show()

,revenue,orders,customers,revenue_share_%,avg_order_value
Country,,,,,
United Kingdom,8282078.723,17901,3942,84.8,463.0
Netherlands,283479.540,93,9,2.9,3048.0
EIRE,259380.020,282,3,2.7,920.0
Germany,200619.660,443,95,2.1,453.0
France,182076.600,382,87,1.9,477.0
Australia,136922.500,56,9,1.4,2445.0
Switzerland,52483.050,50,21,0.5,1050.0
Spain,51746.650,88,30,0.5,588.0
Belgium,36662.960,98,25,0.4,374.0


In [29]:
uk_c = int(country.loc["United Kingdom", "customers"])
all_c = sales.loc[sales["HasCustomerID"], "CustomerID"].nunique()
nl, au, de, fr = (country.loc[c] for c in ["Netherlands", "Australia", "Germany", "France"])
add_insight(f"Countries: the UK accounts for {uk_c:,} of the {all_c:,} identified customers and {country.loc['United Kingdom', 'revenue_share_%']:.1f}% of net product revenue. "
            f"Outside the UK, average order value varies widely: Netherlands ~{nl['avg_order_value']:,.0f} and Australia ~{au['avg_order_value']:,.0f} "
            f"({nl['customers']:.0f} and {au['customers']:.0f} customers) vs Germany ~{de['avg_order_value']:,.0f} and France ~{fr['avg_order_value']:,.0f} "
            f"({de['customers']:.0f} and {fr['customers']:.0f} customers). "
            f"International revenue comes from two patterns: a few large buyers vs many UK-sized buyers. Whether the large ones are wholesalers needs the basket-size analysis.")

**Interpretation.** The UK is 84.8% of net product revenue (3,942 of 4,362 identified customers). Among the top non-UK countries, the picture is not uniform. Netherlands (283k, 9 customers), EIRE (259k, only 3 customers) and Australia (137k, 9 customers) have very large average orders (3,048, 920 and 2,445), so a handful of big accounts drive them. Germany (95 customers) and France (87 customers) have average orders of ~453 and ~477, similar to the UK (463): many ordinary buyers.

**So what:** "international" is not one segment. Treat the few large foreign accounts through account management, and Germany/France as normal customer markets.

### 6.6 Cancellations and returns
**Business intent:** how much revenue do returns cost, and are they a broad behaviour or a few isolated events?

In [30]:
# Cancellations / returns
sold = sales[~sales["IsCancellation"]]
ret  = sales[sales["IsCancellation"]]

returned_value = -ret["TotalLineRevenue"].sum()
print(f"Returned value / gross sales: {returned_value / sold['TotalLineRevenue'].sum():.1%}")
print(f"Cancellation invoices / all invoices: {ret['InvoiceNo'].nunique() / sales['InvoiceNo'].nunique():.1%}")

# --- Products ---
p = pd.DataFrame({
    "sold_units": sold.groupby("StockCode")["Quantity"].sum(),
    "returned_units": -ret.groupby("StockCode")["Quantity"].sum(),
    "returned_value": -ret.groupby("StockCode")["TotalLineRevenue"].sum(),
}).fillna(0)
p["name"] = name_of
p["return_rate"] = (p["returned_units"] / p["sold_units"]).round(3)

print("\nTop 10 products by returned value:")
display(p.nlargest(10, "returned_value"))

# return RATE only for products with enough sales (avoids tiny-denominator noise)
big = p[p["sold_units"] >= 1000]
top_rate = big.nlargest(10, "return_rate").iloc[::-1]
fig = px.bar(top_rate, x="return_rate", y="name", orientation="h", hover_data=["sold_units", "returned_units"],
             title="Highest return rate (products with 1,000+ units sold)", color_discrete_sequence=["#FF2E63"])
fig.show()

# --- Customers (identifiable only) ---
c = pd.DataFrame({
    "bought": sold[sold["HasCustomerID"]].groupby("CustomerID")["TotalLineRevenue"].sum(),
    "returned": -ret[ret["HasCustomerID"]].groupby("CustomerID")["TotalLineRevenue"].sum(),
}).fillna(0)
print(f"\nCustomers with at least one return: {(c['returned'] > 0).mean():.1%}")
print(f"Top 10 customers' share of returned value (identified): {c['returned'].nlargest(10).sum() / c['returned'].sum():.1%}")

Returned value / gross sales: 4.6%
Cancellation invoices / all invoices: 14.8%

Top 10 products by returned value:


,sold_units,returned_units,returned_value,name,return_rate
StockCode,,,,,
23843,80995.0,80995.0,168469.60,"PAPER CRAFT , LITTLE BIRDIE",1.000
23166,78033.0,74494.0,77479.64,MEDIUM CERAMIC TOP STORAGE JAR,0.955
22423,13851.0,855.0,9697.05,REGENCY CAKESTAND 3 TIER,0.062
85123A,37641.0,2578.0,6624.30,WHITE HANGING HEART T-LIGHT HOLDER,0.068
21108,10241.0,3150.0,6591.42,FAIRY CAKE FLANNEL ASSORTED COLOUR,0.308
23113,1154.0,946.0,4803.06,PANTRY CHOPPING BOARD,0.820
48185,3615.0,674.0,4554.90,DOORMAT FAIRY CAKE,0.186
21175,12207.0,2030.0,3775.33,GIN + TONIC DIET METAL SIGN,0.166
47566B,2379.0,1424.0,3692.95,TEA TIME PARTY BUNTING,0.599



Customers with at least one return: 35.2%
Top 10 customers' share of returned value (identified): 67.0%


In [31]:
# 1) How much money did returns cost? (refund value only: we have no cost-of-goods data)
gross_sold = sold["TotalLineRevenue"].sum()
returned_total = -ret["TotalLineRevenue"].sum()
print(f"Returned value: {returned_total:,.0f} = {returned_total / gross_sold:.1%} of gross sales ({gross_sold:,.0f})")

# 2) Products where almost everything sold was also cancelled
mega = p[(p["return_rate"] > 0.9) & (p["returned_value"] > 50000)].index.tolist()
mega_value = p.loc[mega, "returned_value"].sum()
print(f"\nFull-reversal products: {mega} -> {mega_value:,.0f} ({mega_value / returned_total:.1%} of all returned value)")

rest = returned_total - mega_value
gross_rest = gross_sold - sold.loc[sold["StockCode"].isin(mega), "TotalLineRevenue"].sum()
print(f"Returned value without them: {rest:,.0f} = {rest / gross_rest:.1%} of gross sales")

# 3) Evidence: the huge cancellations and the huge sales behind them
cols = ["InvoiceNo", "StockCode", "Quantity", "InvoiceDate", "CustomerID", "MatchStatus"]
print("\nBig cancellations:")
big_c = ret.loc[ret["StockCode"].isin(mega) & (ret["Quantity"] < -10000), cols]
display(big_c)
print("Big sales of the same products:")
big_s = sold.loc[sold["StockCode"].isin(mega) & (sold["Quantity"] > 10000), cols]
display(big_s)

# 4) Test your hypothesis: do pricier products get returned less? (by value, without the full reversals)
pp = sold.groupby("StockCode").agg(sold_value=("TotalLineRevenue", "sum"), sold_units=("Quantity", "sum"))
pp["returned_value"] = -ret.groupby("StockCode")["TotalLineRevenue"].sum()
pp = pp.fillna(0)
pp["avg_price"] = pp["sold_value"] / pp["sold_units"]
pp = pp[(pp["sold_units"] >= 200) & ~pp.index.isin(mega)]
pp["price_band"] = pd.qcut(pp["avg_price"], 4, labels=["Q1 cheapest", "Q2", "Q3", "Q4 priciest"])

band = pp.groupby("price_band", observed=True).agg(
    products=("sold_value", "size"), sold_value=("sold_value", "sum"), returned_value=("returned_value", "sum"))
band["return_rate_value_%"] = (band["returned_value"] / band["sold_value"] * 100).round(2)
band

Returned value: 475,901 = 4.6% of gross sales (10,247,908)

Full-reversal products: ['23166', '23843'] -> 245,949 (51.7% of all returned value)
Returned value without them: 229,952 = 2.3% of gross sales

Big cancellations:


,InvoiceNo,StockCode,Quantity,InvoiceDate,CustomerID,MatchStatus
61011,C541433,23166,-74215,2011-01-18 10:17:00,12346,matched (full qty)
535161,C581484,23843,-80995,2011-12-09 09:27:00,16446,matched (full qty)


Big sales of the same products:


,InvoiceNo,StockCode,Quantity,InvoiceDate,CustomerID,MatchStatus
61006,541431,23166,74215,2011-01-18 10:01:00,12346,not a cancellation
535160,581483,23843,80995,2011-12-09 09:15:00,16446,not a cancellation


,products,sold_value,returned_value,return_rate_value_%
price_band,,,,
Q1 cheapest,583,860551.36,11520.38,1.34
Q2,582,1456409.17,23459.68,1.61
Q3,582,3249340.90,73410.02,2.26
Q4 priciest,583,4085203.89,109379.76,2.68


In [32]:
# Generalise: cancellation lines that reverse an identical purchase (same customer, product, quantity) within 60 minutes
c2 = ret.loc[ret["HasCustomerID"], ["InvoiceNo", "CustomerID", "StockCode", "Quantity", "InvoiceDate", "TotalLineRevenue"]].copy()
c2["q"] = -c2["Quantity"]
b2 = sold.loc[sold["HasCustomerID"], ["CustomerID", "StockCode", "Quantity", "InvoiceDate"]]

mm = c2.merge(b2, left_on=["CustomerID", "StockCode", "q"], right_on=["CustomerID", "StockCode", "Quantity"], suffixes=("", "_buy"))
mm["gap_min"] = (mm["InvoiceDate"] - mm["InvoiceDate_buy"]).dt.total_seconds() / 60
quick = mm[(mm["gap_min"] >= 0) & (mm["gap_min"] <= 60)].drop_duplicates(["InvoiceNo", "StockCode"])

q_value = -quick["TotalLineRevenue"].sum()
print(f"Quick reversals: {len(quick)} of {len(c2)} identified cancellation lines")
print(f"Their value: {q_value:,.0f} = {q_value / returned_total:.1%} of returned value")
print(f"Without the two giant ones: {-quick[~quick['StockCode'].isin(mega)]['TotalLineRevenue'].sum():,.0f}")

# Flag the cancellation lines (we keep them: revenue still nets out)
keys = set(zip(quick["InvoiceNo"], quick["StockCode"]))
df["IsQuickReversal"] = [k in keys for k in zip(df["InvoiceNo"], df["StockCode"])]

log_decision("Quick reversals", "Flagged cancellation lines that reverse an identical purchase within 60 min (IsQuickReversal); kept in df",
             int(df["IsQuickReversal"].sum()),
             "Net revenue unchanged, but return metrics are reported with and without them. Two orders (~80k units each) make up ~52% of returned value")

# Minutes between each giant purchase and its cancellation (same customer + product)
gaps = [(r.InvoiceDate - big_s[(big_s["CustomerID"] == r.CustomerID) & (big_s["StockCode"] == r.StockCode)]["InvoiceDate"].iloc[0]).total_seconds() / 60
        for r in big_c.itertuples()]
units_k = (-big_c["Quantity"] / 1000).round().astype(int)
add_insight(f"Returns: returned value is {returned_total / gross_sold:.1%} of gross product sales and {ret['InvoiceNo'].nunique() / sales['InvoiceNo'].nunique():.1%} of invoices are cancellations. "
            f"Two orders (StockCodes {' and '.join(mega)} by customers {' and '.join(big_c['CustomerID'])}) of ~{units_k.min()}-{units_k.max()}k units were cancelled "
            f"{min(gaps):.0f}-{max(gaps):.0f} minutes after being placed; they are {mega_value / returned_total:.1%} of returned value. "
            f"Without them returns are {rest / gross_rest:.1%} of gross sales.")
add_insight(f"Returns vs price: the value return rate rises with price ({band['return_rate_value_%'].iloc[0]:.2f}% in the cheapest quartile to {band['return_rate_value_%'].iloc[-1]:.2f}% in the priciest, "
            f"excluding the two reversals). Descriptive only: the data does not support 'pricier products are returned less'. Product type or buyer type may explain it.")
print(len(insights), "insights so far")

Quick reversals: 422 of 8506 identified cancellation lines
Their value: 278,694 = 58.6% of returned value
Without the two giant ones: 33,026
7 insights so far


**Interpretation.** Returns cost 4.6% of gross sales (476k of 10.25M); 14.8% of invoices are cancellations and 35.2% of identified customers have at least one return. Ten customers account for 67.0% of returned value. But the headline is distorted by two events: two orders of ~74k and ~81k units (StockCodes 23166 and 23843) were cancelled 12 and 16 minutes after being placed. They are **51.7% of returned value; without them returns are 2.3% of gross sales.** The return-rate chart shows the same artefacts at the top (23843 at 100%, 23166 at 95.5%), plus other near-full reversals worth checking (e.g. Rotating Silver Angels T-Light Holder at 99%, 9,461 units sold).

More generally, 422 of 8,506 identified cancellation lines reverse an identical purchase within 60 minutes (278,694, or 58.6% of returned value). We treat them as mistaken orders, not customer returns. 84.3% of all cancellation lines match an earlier full-quantity purchase; 14.7% have no original in the data (probably bought before Dec 2010).

The value return rate rises with price (1.34% in the cheapest quartile to 2.68% in the priciest). This is descriptive only; the data cannot say why.

**So what:** a check on unusually large quantities at order time is a hypothesis worth testing; day-to-day returns are a modest 2.3% of sales.

### 6.7 Basket behaviour: wholesale-like orders?
**Business intent:** AOV and basket size per *invoice* (not per row) to tell bulk from individual purchasing. Orders reversed within 60 minutes are removed from this view (they are mistakes, see 6.6).

In [33]:
# Basket behaviour: one row per invoice (purchases only), to spot wholesale-like orders
inv = buys.groupby("InvoiceNo").agg(
    items=("Quantity", "sum"),
    lines=("StockCode", "nunique"),
    value=("TotalLineRevenue", "sum"),
    country=("Country", "first"),
)
print(inv[["value", "items", "lines"]].describe(percentiles=[.5, .75, .9, .99]).round(1))

fig = px.histogram(inv[inv["value"] > 0], x="value", nbins=60, log_y=True,
                   title="Invoice value distribution (log count axis)", color_discrete_sequence=["#08D9D6"])
fig.show()

s_inv = inv["value"].sort_values(ascending=False)
print(f"\nMean vs median order value: {inv['value'].mean():,.0f} vs {inv['value'].median():,.0f}")
print(f"Top 10% of invoices share of revenue: {s_inv.head(int(len(s_inv) * 0.1)).sum() / s_inv.sum():.1%}")

          value    items    lines
count   19773.0  19773.0  19773.0
mean      518.3    281.3     26.2
std      1764.2    958.8     47.0
min         0.4      1.0      1.0
50%       302.2    152.0     15.0
75%       481.5    297.0     29.0
90%       915.8    531.0     52.0
99%      4408.9   2232.8    218.0
max    168469.6  80995.0   1109.0



Mean vs median order value: 518 vs 302
Top 10% of invoices share of revenue: 48.8%


In [34]:
# Basket view without orders that were reversed within 60 minutes (the giant mistaken orders)
b3 = sold.loc[sold["HasCustomerID"], ["CustomerID", "StockCode", "Quantity", "InvoiceDate"]].reset_index().rename(columns={"index": "buy_idx"})
c3 = ret.loc[ret["HasCustomerID"], ["CustomerID", "StockCode", "Quantity", "InvoiceDate"]].copy()
c3["q"] = -c3["Quantity"]
m3 = c3.merge(b3, left_on=["CustomerID", "StockCode", "q"], right_on=["CustomerID", "StockCode", "Quantity"], suffixes=("_c", "_b"))
m3["gap_min"] = (m3["InvoiceDate_c"] - m3["InvoiceDate_b"]).dt.total_seconds() / 60
rev_idx = m3.loc[m3["gap_min"].between(0, 60), "buy_idx"].unique()
buys_clean = buys.drop(index=buys.index.intersection(rev_idx))
print("Purchase lines removed from the basket view:", len(buys) - len(buys_clean))

# One row per invoice
inv = buys_clean.groupby("InvoiceNo").agg(
    items=("Quantity", "sum"), lines=("StockCode", "nunique"),
    value=("TotalLineRevenue", "sum"), country=("Country", "first"),
    customer=("CustomerID", "first"), is_uk=("Is_UK", "first"),
)
inv = inv[inv["value"] > 0]
inv["units_per_line"] = inv["items"] / inv["lines"]

print(inv[["value", "items", "lines", "units_per_line"]].describe(percentiles=[.5, .75, .9, .99]).round(1))
print(f"\nMean vs median order value: {inv['value'].mean():,.0f} vs {inv['value'].median():,.0f}")

# Wholesale-like behaviour. Declared rule (adjustable): 'bulk-like' invoice = value in the top 10%
cut = inv["value"].quantile(0.90)
inv["bulk_like"] = inv["value"] >= cut
print(f"Bulk-like threshold: invoice value >= {cut:,.0f}")

bulk_prof = inv.groupby("bulk_like").agg(
    invoices=("value", "size"), revenue=("value", "sum"), median_value=("value", "median"),
    median_items=("items", "median"), median_lines=("lines", "median"),
    median_units_per_line=("units_per_line", "median"), uk_share=("is_uk", "mean"),
)
bulk_prof["revenue_%"] = (bulk_prof["revenue"] / bulk_prof["revenue"].sum() * 100).round(1)
display(bulk_prof.round(2))

smp = inv.sample(5000, random_state=SEED)
fig = px.scatter(smp, x="items", y="value", color="bulk_like", log_x=True, log_y=True,
                 hover_data=["country", "lines"], title="Invoices: units vs value (log scales, 5,000 sampled)",
                 color_discrete_map={False: "#08D9D6", True: "#FF2E63"})
fig.show()

ci = inv[inv["customer"].notna()].groupby("customer")["bulk_like"].mean()
print(f"\nIdentified customers with at least one bulk-like invoice: {(ci > 0).mean():.1%}")
print(f"Customers whose invoices are ALL bulk-like: {(ci == 1).mean():.1%}")

Purchase lines removed from the basket view: 472
         value    items    lines  units_per_line
count  19711.0  19711.0  19711.0         19711.0
mean     503.5    273.2     26.2            26.5
std     1146.3    559.3     47.1           104.6
min        0.4      1.0      1.0             1.0
50%      302.2    152.0     15.0             9.9
75%      481.2    297.0     29.0            16.7
90%      911.2    530.0     52.0            41.8
99%     4391.1   2223.1    218.0           360.0
max    52940.9  15049.0   1109.0          4300.0

Mean vs median order value: 504 vs 302
Bulk-like threshold: invoice value >= 911


,invoices,revenue,median_value,median_items,median_lines,median_units_per_line,uk_share,revenue_%
bulk_like,,,,,,,,
False,17739,5227408.21,270.25,134.0,14.0,9.52,0.92,52.7
True,1972,4697474.28,1498.09,706.0,47.0,15.65,0.82,47.3



Identified customers with at least one bulk-like invoice: 12.8%
Customers whose invoices are ALL bulk-like: 2.6%


**Interpretation.** Invoice values are right-skewed: mean 504 vs median 302, and the histogram (log count axis) shows a long tail. The top 10% of invoices (value >= ~911) bring **47.3% of revenue**. They differ from the rest on every measure: median value 1,498 vs 270, 706 vs 134 items, 47 vs 14 product lines, and 15.7 vs 9.5 units per line. In the scatter, units and value grow together and the bulk-like invoices sit top-right.

But 82% of these invoices are still UK, so this is a **bulk-like signal, not proof of wholesale**. At customer level, 12.8% of identified customers placed at least one bulk-like invoice, and only 2.6% have *all* invoices bulk-like: a small group of trade-like accounts, plus regular customers who occasionally place big orders. Removing the 472 lines of mistaken, quickly reversed orders lowers the largest invoice from 168k to 53k.

### 6.8 What does a typical customer look like?
**Business intent:** describe the 'median' customer before segmenting.

In [35]:
# What does a typical customer look like? (identifiable customers, purchases only, reversed orders removed)
bc = buys_clean[buys_clean["HasCustomerID"]]
orders_eda = bc.groupby(["CustomerID", "InvoiceNo"]).agg(
    value=("TotalLineRevenue", "sum"), units=("Quantity", "sum"), date=("InvoiceDate", "min")).reset_index()
orders_eda = orders_eda[orders_eda["value"] > 0].sort_values(["CustomerID", "date"])
orders_eda["gap_days"] = orders_eda.groupby("CustomerID")["date"].diff().dt.days

cf = orders_eda.groupby("CustomerID").agg(
    n_orders=("InvoiceNo", "nunique"), total_value=("value", "sum"),
    avg_order_value=("value", "mean"), avg_units=("units", "mean"),
    first=("date", "min"), last=("date", "max"),
)
cf["active_days"] = (cf["last"] - cf["first"]).dt.days
cf["median_gap_days"] = orders_eda.groupby("CustomerID")["gap_days"].median()

print("Customers:", len(cf))
display(cf[["n_orders", "total_value", "avg_order_value", "avg_units", "active_days", "median_gap_days"]]
        .describe(percentiles=[.5, .75, .9]).round(1))
print(f"One-time buyers: {(cf['n_orders'] == 1).mean():.1%} | repeat buyers: {(cf['n_orders'] > 1).mean():.1%}")

fig = px.histogram(cf[cf["n_orders"] <= 30], x="n_orders", nbins=30,
                   title="Orders per customer (capped at 30 for readability)", color_discrete_sequence=["#08D9D6"])
fig.show()

Customers: 4333


,n_orders,total_value,avg_order_value,avg_units,active_days,median_gap_days
count,4333.0,4333.0,4333.0,4333.0,4333.0,2825.0
mean,4.2,1941.9,375.8,229.0,130.2,68.6
std,7.6,8399.3,474.7,311.5,132.1,67.2
min,1.0,2.9,2.9,1.0,0.0,0.0
50%,2.0,658.9,289.5,161.0,91.0,47.0
75%,5.0,1624.0,423.8,271.9,252.0,89.0
90%,9.0,3540.3,656.1,420.5,338.8,159.0
max,203.0,279138.0,14844.8,7824.0,373.0,365.0


One-time buyers: 34.8% | repeat buyers: 65.2%


**Interpretation.** The typical customer is much smaller than the average: median 2 orders and 659 total spend (mean 4.2 orders and 1,942), median order value 290, ~161 units per order, and a median active span of 91 days. 34.8% bought once, 80.1% placed 5 orders or fewer, and only 8.9% placed 10 or more. Repeat buyers reorder after a median of 47 days.

**So what:** averages are pulled up by a few heavy buyers, so segment profiles in this notebook use medians next to means.

### 6.9 Product concentration (Pareto)
**Business intent:** check whether 20% of products generate 80% of revenue.

In [36]:
# Pareto: do 20% of products generate 80% of revenue? (net revenue per StockCode)
pr = prod["revenue"].sort_values(ascending=False)
pr = pr[pr > 0]
cum = pr.cumsum() / pr.sum()
x = np.arange(1, len(pr) + 1) / len(pr)
n80 = int((cum < 0.8).sum() + 1)

print(f"Products with positive net revenue: {len(pr)} (excluded {len(prod) - len(pr)} with zero/negative)")
print(f"{n80} products ({n80 / len(pr):.1%}) generate 80% of revenue")
print(f"Top 20% of products generate {cum.iloc[int(len(pr) * 0.2) - 1]:.1%} of revenue")

fig = go.Figure(go.Scatter(x=x * 100, y=cum.values * 100, mode="lines", line=dict(color="#FF2E63")))
fig.add_hline(y=80, line_dash="dash")
fig.add_vline(x=n80 / len(pr) * 100, line_dash="dash")
fig.update_layout(title="Cumulative revenue by product (Pareto curve)",
                  xaxis_title="% of products (ranked by revenue)", yaxis_title="% of cumulative revenue")
fig.show()

Products with positive net revenue: 3905 (excluded 23 with zero/negative)
841 products (21.5%) generate 80% of revenue
Top 20% of products generate 78.3% of revenue


**Interpretation.** Revenue is concentrated, but less than the textbook rule: 841 products (21.5%) generate 80% of net revenue, and the top 20% generate 78.3%. No single product dominates (the top 10 products are 8.2% of revenue, the top 1% are 17.8%).

**So what:** assortment decisions should look at the long tail, but ~20% of revenue still comes from the ~78% of products outside the core range.

### 6.10 Dead stock
**Business intent:** find products with high historical volume but no sales in the final 90 days. Declared rule: top 25% by historical units and zero or negative units in the last 90 days.

In [37]:
# Dead stock: high historical volume but zero/negative net sales in the last 90 days
max_date = sales["InvoiceDate"].max()
cutoff = max_date - pd.Timedelta(days=90)
print("Last-90-days window:", cutoff.date(), "->", max_date.date())

before, after = sales[sales["InvoiceDate"] < cutoff], sales[sales["InvoiceDate"] >= cutoff]
ds = pd.DataFrame({
    "hist_units": before.groupby("StockCode")["Quantity"].sum(),
    "hist_revenue": before.groupby("StockCode")["TotalLineRevenue"].sum(),
})
ds["last90_units"] = after.groupby("StockCode")["Quantity"].sum()
ds["last90_units"] = ds["last90_units"].fillna(0)
ds["name"] = name_of

high = ds["hist_units"] >= ds["hist_units"].quantile(0.75)      # declared rule: top 25% by historical units
dead = ds[high & (ds["last90_units"] <= 0)].sort_values("hist_units", ascending=False)

print(f"High-volume products: {high.sum()} | dead in the last 90 days: {len(dead)} ({len(dead) / high.sum():.1%})")
print(f"Their share of historical revenue: {dead['hist_revenue'].sum() / ds['hist_revenue'].sum():.1%}")
dead.head(15)

Last-90-days window: 2011-09-10 -> 2011-12-09
High-volume products: 921 | dead in the last 90 days: 22 (2.4%)
Their share of historical revenue: 0.7%


,hist_units,hist_revenue,last90_units,name
StockCode,,,,
18007,5884,395.61,0.0,ESSENTIAL BALM 3.5g TIN IN ENVELOPE
10133,2846,1535.40,0.0,COLOURING PENCILS BROWN TUBE
22855,2630,3286.73,0.0,FINE WICKER HEART
47556B,2600,6045.00,0.0,TEA TIME TEA TOWELS
17096,2545,455.98,0.0,ASSORTED LAQUERED INCENSE HOLDERS
21399,1943,826.32,0.0,BLUE POLKADOT COFFEE MUG
21033,1931,4849.94,0.0,JUMBO BAG CHARLIE AND LOLA TOYS
22088,1837,5511.85,0.0,PAPER BUNTING COLOURED LACE
21398,1532,729.60,0.0,RED POLKADOT COFFEE MUG


**Interpretation.** Dead stock is a small issue: 22 of 921 high-volume products (2.4%) had no net sales in the last 90 days (10 Sep to 9 Dec 2011), and together they represent only 0.7% of historical revenue. The largest by units are low-value items (e.g. Essential Balm tin: 5,884 units but only 396 revenue). Some may be seasonal, since the window ends on 9 Dec.

**So what:** clearance is not a priority; check seasonality before delisting any of these.

### 6.11 Weekday x hour heatmap
**Business intent:** one view of when revenue happens.

In [38]:
# Weekday x hour revenue (purchases only, interactive)
hm = buys_clean.pivot_table(index="DayOfWeek", columns="Hour", values="TotalLineRevenue", aggfunc="sum")
hm = hm.reindex(["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Sunday"])   # no Saturday in the data

fig = px.imshow(hm, aspect="auto", color_continuous_scale=["#EAEAEA", "#08D9D6", "#FF2E63"],
                title="Revenue by weekday and hour", labels=dict(color="Revenue"))
fig.show()

**Interpretation.** There is no single hot spot, but a broad plateau on Tuesday-Thursday between 10:00 and 15:00. The strongest slots are almost tied: Thursday 10:00 (291k), Tuesday 15:00 (290k) and Wednesday 12:00 (288k). Sunday is clearly weakest and there is no Saturday row at all.

**So what:** ad timing and maintenance windows should be planned for the whole Tuesday-Thursday daytime block, not for a single peak hour.

## 7. Business Insights (10+)

**Business intent:** collect the findings in one register, each backed by a number computed in code. Insights are descriptive; none of them claims a cause. The register below shows the insights from the EDA; the RFM and segmentation insights are added in sections 9 and 13 and the full list is printed in section 14.

In [39]:
top20_share = cum.iloc[int(len(pr) * 0.2) - 1]
add_insight(f"Pareto: {n80:,} of {len(pr):,} products ({n80 / len(pr):.1%}) generate 80% of net revenue; the top 20% generate {top20_share:.1%}. "
            f"Revenue is concentrated in a minority of SKUs, with a long tail of low-contribution products.")

bk, nb_ = bulk_prof.loc[True], bulk_prof.loc[False]
add_insight(f"Baskets: the top 10% of invoices (value >= ~{cut:,.0f}) generate {bk['revenue_%']:.1f}% of revenue, with median value ~{bk['median_value']:,.0f} vs ~{nb_['median_value']:,.0f} for the rest. "
            f"This is a bulk-like signal, not proof of wholesale: {bk['uk_share']:.0%} of these invoices are still UK.")

add_insight(f"Dead stock: {len(dead)} of {high.sum()} high-volume products ({len(dead) / high.sum():.1%}) had zero sales in the last 90 days, but they represent only "
            f"{dead['hist_revenue'].sum() / ds['hist_revenue'].sum():.1%} of historical revenue. The issue is small; some may be seasonal since the window and the data end on 9 Dec.")

hm_s = hm.stack().sort_values(ascending=False)
(pk_day, pk_hour), pk_val = hm_s.index[0], hm_s.iloc[0]
tt = hm.loc[["Tuesday", "Thursday"]].sum().sum() / hm.sum().sum()
add_insight(f"Heatmap: the single strongest slot is {pk_day} at {pk_hour}:00 ({pk_val/1e3:,.0f}k, {pk_val / hm_s.sum():.1%} of purchase revenue); "
            f"Tuesday and Thursday together carry {tt:.0%} of weekday-hour revenue. Maintenance should avoid these peaks, and ad timing can be tested around them.")

In [40]:
for i, t in enumerate(insights, 1):
    print(f"{i}. {t}\n")

1. Products: only 5 of the top-10 products by units are also in the top-10 by revenue. Popcorn Holder leads in units (~56k) but ranks 10 by revenue; Regency Cakestand 3 Tier leads revenue (~164k) but ranks 44 by units. Price per unit, not just volume, decides what drives revenue.

2. Time: revenue from Sep-Nov 2011 is ~3.5M, about 36% of the whole period, and Nov 2011 (1.43M) is ~2.3x the Jan-Aug monthly average (~634k). Descriptive only: with ~1 year of data we cannot call it a recurring seasonal pattern, and Dec 2011 is incomplete (ends on the 9th).

3. Customers: the top 20% of identifiable customers generate 74% of identified revenue, and 30.4% of customers bought only once. Revenue is concentrated, and a large share of customers never came back in the observed period.

4. Hours/days: revenue peaks in the hours 10:00-15:59 (top hours: 12, 10, 15, 13, 11) and no transactions are recorded on Saturday.

5. Countries: the UK accounts for 3,942 of the 4,362 identified customers and 84.8

## 8. Customer-Level Feature Engineering

**Business intent:** one row per identifiable customer, built from clean purchases (mistaken, quickly reversed orders removed), so RFM and clustering describe real behaviour and not data artefacts. `CustomerID` is the index, never a feature.

In [41]:
# Customer-level feature engineering
# Business intent: one row per identifiable customer, built from clean purchases (reversed mega-orders removed),
# so that RFM and clustering describe real customer behaviour, not data artefacts.

# Valid orders per customer (purchases only, positive value)
orders = (buys_clean[buys_clean["HasCustomerID"]]
          .groupby(["CustomerID", "InvoiceNo"])
          .agg(value=("TotalLineRevenue", "sum"),
               units=("Quantity", "sum"),
               lines=("StockCode", "nunique"),
               date=("InvoiceDate", "min"))
          .reset_index())
orders = orders[orders["value"] > 0].sort_values(["CustomerID", "date"])
orders["gap_days"] = orders.groupby("CustomerID")["date"].diff().dt.days

# Returns per customer. Cancellations that reversed a mistaken order within 60 min (`keys`, section 6.6) are not
# customer returns: their purchase was already removed from spend in buys_clean.
ret_clean = ret[[(i, s) not in keys for i, s in zip(ret["InvoiceNo"], ret["StockCode"])]]
cust_ret = (-ret_clean[ret_clean["HasCustomerID"]].groupby("CustomerID")["TotalLineRevenue"].sum()).rename("returned_value")

# Assemble the customer table
cust = orders.groupby("CustomerID").agg(
    n_orders=("InvoiceNo", "nunique"),
    total_spend=("value", "sum"),
    avg_order_value=("value", "mean"),
    avg_units_per_order=("units", "mean"),
    avg_lines_per_order=("lines", "mean"),
    first_purchase=("date", "min"),
    last_purchase=("date", "max"),
    median_gap_days=("gap_days", "median"),
)
cust["returned_value"] = cust_ret.reindex(cust.index).fillna(0)
# Capped at 1: a customer can return more than they bought in-window if the purchase was before Dec 2010
cust["return_ratio"] = (cust["returned_value"] / cust["total_spend"]).clip(upper=1).round(3)
cust["tenure_days"] = (cust["last_purchase"] - cust["first_purchase"]).dt.days

# Country: one per customer (most frequent)
cust["country"] = buys_clean[buys_clean["HasCustomerID"]].groupby("CustomerID")["Country"].agg(lambda s: s.mode().iloc[0])

print("Customers:", len(cust))
display(cust.describe().round(1).T)
print("\nCustomers with returns:", (cust["returned_value"] > 0).sum())

Customers: 4333


,count,mean,min,25%,50%,75%,max,std
n_orders,4333.0,4.2,1.0,1.0,2.0,5.0,203.0,7.6
total_spend,4333.0,1941.9,2.9,303.2,658.9,1624.0,279138.0,8399.3
avg_order_value,4333.0,375.8,2.9,176.8,289.5,423.8,14844.8,474.7
avg_units_per_order,4333.0,229.0,1.0,92.5,161.0,271.9,7824.0,311.5
avg_lines_per_order,4333.0,21.6,1.0,9.4,16.8,27.8,297.9,19.5
first_purchase,4333,2011-04-30 17:44:22.654050304,2010-12-01 08:26:00,2011-01-17 11:12:00,2011-04-05 12:24:00,2011-08-19 10:51:00,2011-12-09 12:16:00,NaN
last_purchase,4333,2011-09-08 05:51:37.858296832,2010-12-01 09:53:00,2011-07-20 10:15:00,2011-10-20 09:32:00,2011-11-22 10:44:00,2011-12-09 12:50:00,NaN
median_gap_days,2825.0,68.6,0.0,24.0,47.0,89.0,365.0,67.2
returned_value,4333.0,43.6,0.0,0.0,0.0,12.8,22998.4,440.8
return_ratio,4333.0,0.0,0.0,0.0,0.0,0.0,1.0,0.1



Customers with returns: 1461


In [42]:
dropped = set(sales.loc[sales["HasCustomerID"], "CustomerID"]) - set(cust.index)
print("Customers in EDA but not in the customer table:", len(dropped))

d = sales[sales["CustomerID"].isin(dropped)].groupby("CustomerID").agg(
    rows=("InvoiceNo", "size"), cancel_rows=("IsCancellation", "sum"), net_value=("TotalLineRevenue", "sum"))
d.sort_values("rows", ascending=False).head(10)

Customers in EDA but not in the customer table: 29


,rows,cancel_rows,net_value
CustomerID,,,
16252,21,21,-295.09
15823,15,15,-811.86
14627,5,5,-21.85
14213,5,5,-1192.20
13958,5,5,-102.45
17820,5,5,-24.05
15728,4,4,-134.80
13693,4,4,-32.00
18256,4,4,-50.10


**Interpretation.** The customer table has 4,333 customers. Medians are far below means (spend 659 vs 1,942; orders 2 vs 4.2), and the maximum spend is 279k, so a few extreme customers exist. 1,461 customers have returns, but the return ratio is small for most (mean ~2%, capped at 1). 29 customers were left out because they have no valid purchase in the period (the ten largest are all cancellation lines only), so there is nothing to describe.

## 9. RFM Analysis

**Business intent:** translate transactions into three dimensions. **Recency** = days since the last purchase, **Frequency** = number of unique valid invoices, **Monetary** = sum of purchase value. Snapshot date = the day after the last transaction.

### 9.1 Scoring (1-5)
**Direction of the scores:** for R, *fewer days since the last purchase = higher score* (5 = most recent). For F and M, *more = higher score*. The string `RFM_Score` (e.g. `555`) is R, F, M in that order.

In [43]:
# RFM
# Business intent: Recency = how recently, Frequency = how often, Monetary = how much.

snapshot = sales["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)
print("Snapshot date:", snapshot.date())

rfm = pd.DataFrame({
    "Recency":   (snapshot - cust["last_purchase"]).dt.days,
    "Frequency": cust["n_orders"],
    "Monetary":  cust["total_spend"],
})

# Scoring: 5 quantiles. For Recency, LOWER days = HIGHER score (5 = most recent).
rfm["R"] = pd.qcut(rfm["Recency"], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F"] = pd.cut(rfm["Frequency"], bins=[0, 1, 2, 4, 8, np.inf], labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M"] = pd.qcut(rfm["Monetary"], 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["RFM_Score"] = rfm["R"].astype(str) + rfm["F"].astype(str) + rfm["M"].astype(str)

log_decision("RFM scoring", "R and M scored with pd.qcut (5 quantiles). F scored with fixed bins [1, 2, 3-4, 5-8, 9+] because qcut cannot split F: about a third of customers have exactly 1 order",
             len(rfm), "F groups are not equal-sized (see 9.2), but the score keeps its direction: higher F = more orders")
display(rfm.describe().round(1).T)
print(rfm["RFM_Score"].value_counts().head(10))

Snapshot date: 2011-12-10


,count,mean,std,min,25%,50%,75%,max
Recency,4333.0,92.3,100.2,0.0,17.0,50.0,142.0,373.0
Frequency,4333.0,4.2,7.6,1.0,1.0,2.0,5.0,203.0
Monetary,4333.0,1941.9,8399.3,2.9,303.2,658.9,1624.0,279138.0
R,4333.0,3.0,1.4,1.0,2.0,3.0,4.0,5.0
F,4333.0,2.5,1.4,1.0,1.0,2.0,4.0,5.0
M,4333.0,3.0,1.4,1.0,2.0,3.0,4.0,5.0


RFM_Score
111    329
555    267
112    209
211    146
311    133
212    111
545    106
312    102
544    101
455     98
Name: count, dtype: int64


**Interpretation.** The most common RFM scores are the two extremes: `111` (329 customers: oldest, one order, lowest spend) and `555` (267 customers: recent, frequent, top spend). The base is polarised between very good and very weak customers, with fewer in the middle.

### 9.2 Sanity checks
**Business intent:** the scores must behave as documented. If any check fails, every later segment would be wrong.

In [44]:
# Sanity checks: the scores must behave as documented
assert len(rfm) == len(cust) and rfm[["R","F","M"]].notna().all().all()

chk = rfm.groupby("R")["Recency"].mean().round(0)
print("Avg Recency days by R score (must DECREASE as R grows):\n", chk)
print("\nCustomers per F score:\n", rfm["F"].value_counts().sort_index())
print("\nCustomers per M score:\n", rfm["M"].value_counts().sort_index())
print("\nShare of one-time buyers inside F=1:", f"{(rfm.loc[rfm['F']==1,'Frequency']==1).mean():.0%}")

Avg Recency days by R score (must DECREASE as R grows):
 R
1    269.0
2    115.0
3     51.0
4     23.0
5      6.0
Name: Recency, dtype: float64

Customers per F score:
 F
1    1508
2     831
3     899
4     643
5     452
Name: count, dtype: int64

Customers per M score:
 M
1    867
2    866
3    867
4    866
5    867
Name: count, dtype: int64

Share of one-time buyers inside F=1: 100%


**Interpretation.** All checks pass. Recency goes down as R goes up (269 days at R=1, 6 days at R=5), so R=5 does mean most recent. M has equal groups (~866 customers each). **F groups are not equal** (1,508 / 831 / 899 / 643 / 452) because about a third of customers have exactly one order, so quantile cuts cannot split F: fixed bins were used instead (logged in the decision log). F=1 contains only one-time buyers.

### 9.3 Rule-based segments (the guideline's Business Translation Matrix)
Rules are evaluated top to bottom and the first match wins. They follow the guideline (Champions, Loyal, At-Risk High-Value, New Inflow) and add two groups so that every customer has a home (About to Sleep as default, Lost / Hibernating). These are **rule-based labels**; section 12-13 builds data-driven segments and compares both.

In [45]:
# Rule-based segments (mapped to the guideline's Business Translation Matrix).
# Rules are evaluated top-to-bottom; the first match wins.
R, F, M = rfm["R"], rfm["F"], rfm["M"]

conditions = [
    (R >= 4) & (F >= 4) & (M >= 4),          # Champions
    (R <= 2) & (F >= 3) & (M >= 3),          # At-Risk / High-Value Churn
    (R >= 3) & (F >= 3),                     # Loyal Customers
    (R >= 4) & (F <= 2),                     # New Inflow
    (R <= 2) & (F <= 2),                     # Lost / Hibernating
]
labels = ["Champions", "At-Risk High-Value", "Loyal", "New Inflow", "Lost / Hibernating"]
rfm["Segment"] = np.select(conditions, labels, default="About to Sleep")

order = ["Champions", "Loyal", "New Inflow", "About to Sleep", "At-Risk High-Value", "Lost / Hibernating"]
rfm["Segment"] = pd.Categorical(rfm["Segment"], categories=order, ordered=True)
print(rfm["Segment"].value_counts().reindex(order))

Segment
Champions              778
Loyal                  833
New Inflow             518
About to Sleep         542
At-Risk High-Value     319
Lost / Hibernating    1343
Name: count, dtype: int64


### 9.4 Segment profile
**Business intent:** a label means nothing without statistics. Every segment below is described by size, recency, frequency, spend and revenue share.

In [46]:
# Segment profile: every label must be backed by statistics
prof = rfm.join(cust[["avg_order_value", "return_ratio"]]).groupby("Segment", observed=True).agg(
    customers=("Recency", "size"),
    avg_recency_days=("Recency", "mean"),
    median_orders=("Frequency", "median"),
    avg_orders=("Frequency", "mean"),
    median_spend=("Monetary", "median"),
    total_revenue=("Monetary", "sum"),
    avg_order_value=("avg_order_value", "mean"),
    avg_return_ratio=("return_ratio", "mean"),
)
prof["customers_%"] = prof["customers"] / prof["customers"].sum() * 100
prof["revenue_%"] = prof["total_revenue"] / prof["total_revenue"].sum() * 100
prof["revenue_per_customer"] = prof["total_revenue"] / prof["customers"]
display(prof.round(1))

,customers,avg_recency_days,median_orders,avg_orders,median_spend,total_revenue,avg_order_value,avg_return_ratio,customers_%,revenue_%,revenue_per_customer
Segment,,,,,,,,,,,
Champions,778,11.5,9.0,12.6,3006.9,5320729.2,450.9,0.0,18.0,63.2,6839.0
Loyal,833,31.1,4.0,4.4,1116.5,1390544.2,366.4,0.0,19.2,16.5,1669.3
New Inflow,518,16.8,2.0,1.5,362.4,252692.1,316.9,0.0,12.0,3.0,487.8
About to Sleep,542,64.8,1.0,1.6,363.8,270737.0,354.9,0.0,12.5,3.2,499.5
At-Risk High-Value,319,130.0,4.0,4.5,1236.5,567678.9,418.0,0.0,7.4,6.7,1779.6
Lost / Hibernating,1343,208.3,1.0,1.3,307.8,611821.1,359.1,0.0,31.0,7.3,455.6


**Interpretation.** The labels are backed by the numbers. Champions are 18.0% of customers and 63.2% of revenue (median 9 orders, last purchase ~11 days ago, 6,839 per customer), while Lost / Hibernating is 31.0% of customers and only 7.3% of revenue (456 per customer). At-Risk High-Value is 319 customers (7.4%) who placed 4.5 orders on average, spent 1,780 per customer and have been silent for ~130 days: the clearest win-back group in the rule-based view. New Inflow (12.0% of customers, 3.0% of revenue, 1.5 orders on average) are recent and small.

*Caveat:* "Champions" mixes ordinary heavy buyers with extreme accounts; section 13 shows why that matters.

## 10. RFM Visualization & Interpretation

**Business intent:** see where value sits and why the data needs transforming before clustering.

In [47]:
SEG_COLORS = {"Champions": "#FF2E63", "Loyal": "#08D9D6", "New Inflow": "#7BD389",
              "About to Sleep": "#F9C74F", "At-Risk High-Value": "#F3722C", "Lost / Hibernating": "#8D99AE"}

# Raw R, F, M distributions -> justifies the log transform in section 11
fig = make_subplots(rows=1, cols=3, subplot_titles=["Recency (days)", "Frequency (orders)", "Monetary (spend)"])
for i, col in enumerate(["Recency", "Frequency", "Monetary"], 1):
    fig.add_trace(go.Histogram(x=rfm[col], nbinsx=50, marker_color="#08D9D6", showlegend=False), row=1, col=i)
fig.update_layout(title="Raw RFM distributions (heavily right-skewed in F and M)")
fig.show()
print(rfm[["Recency", "Frequency", "Monetary"]].skew().round(2))

# Segment size vs revenue share: who really carries the business?
sp = prof.reset_index()[["Segment", "customers_%", "revenue_%"]].melt("Segment", var_name="metric", value_name="percent")
fig = px.bar(sp, x="Segment", y="percent", color="metric", barmode="group",
             color_discrete_sequence=["#08D9D6", "#FF2E63"], title="Share of customers vs share of revenue by segment")
fig.show()

# Treemap: size = customers, colour = revenue per customer
fig = px.treemap(prof.reset_index(), path=["Segment"], values="customers", color="revenue_per_customer",
                 color_continuous_scale=["#EAEAEA", "#08D9D6", "#FF2E63"],
                 title="Segments: size = customers, colour = revenue per customer")
fig.show()

# R x F heatmap: average spend (shows the F-M link and where value sits)
hm = rfm.pivot_table(index="R", columns="F", values="Monetary", aggfunc="mean").sort_index(ascending=False)
fig = px.imshow(hm.round(0), text_auto=True, aspect="auto", labels=dict(x="F score", y="R score", color="Avg spend"),
                color_continuous_scale=["#EAEAEA", "#08D9D6", "#FF2E63"], title="Average spend by Recency x Frequency score")
fig.show()

# 3D scatter on log axes (sampled for speed)
smp = rfm.sample(min(2500, len(rfm)), random_state=SEED)
fig = px.scatter_3d(smp, x="Recency", y="Frequency", z="Monetary", color="Segment", log_y=True, log_z=True,
                    color_discrete_map=SEG_COLORS, opacity=0.7, title="Customers in R-F-M space (log F and M, 2,500 sampled)")
fig.update_traces(marker_size=3)
fig.show()

# Correlation between log-RFM (are the 3 dimensions redundant?)
corr = np.log1p(rfm[["Recency", "Frequency", "Monetary"]]).corr().round(2)
fig = px.imshow(corr, text_auto=True, color_continuous_scale="RdBu_r", zmin=-1, zmax=1, title="Correlation of log(R), log(F), log(M)")
fig.show()

Recency       1.24
Frequency    11.83
Monetary     21.21
dtype: float64


**Interpretation of the charts above**
- **Distributions:** Frequency (skew 11.8) and Monetary (skew 21.2) are heavily right-skewed: most customers are small and a few are huge. Recency is mildly skewed (1.24). This is why section 11 applies a log transform.
- **Customers vs revenue share:** Champions are 18% of customers but 63% of revenue; Lost / Hibernating are 31% of customers but 7%.
- **Treemap:** size is customers, colour is revenue per customer; Champions are by far the "hottest" box (6,839 per customer).
- **R x F heatmap:** average spend rises mainly with Frequency, not Recency. At F=1 the average is 315-428 for every R; at F=4 it is 1,649-2,469. Only the F=5 row changes with R, and its top cell (R=5, 12,391) is driven by a few extreme accounts.
- **3D scatter:** Champions sit at low recency, high frequency and spend; Lost / Hibernating at high recency with low F and M. The groups touch and overlap, which anticipates the modest silhouette in section 12.
- **Correlation (log RFM):** Frequency and Monetary are strongly related (0.81); Recency is negatively related to both (-0.57 with F, -0.49 with M). F and M carry overlapping information, so clustering effectively works with about two independent dimensions.

In [48]:
ch, lo, ar = prof.loc["Champions"], prof.loc["Lost / Hibernating"], prof.loc["At-Risk High-Value"]
add_insight(f"RFM: Champions are {ch['customers_%']:.1f}% of customers but {ch['revenue_%']:.1f}% of revenue "
            f"(median {ch['median_orders']:.0f} orders, avg recency {ch['avg_recency_days']:.0f} days).")
add_insight(f"RFM: At-Risk High-Value customers ({ar['customers']:.0f}) have spent {ar['revenue_%']:.1f}% of revenue "
            f"but have been inactive for ~{ar['avg_recency_days']:.0f} days on average: the most valuable win-back target.")
add_insight(f"RFM: Lost / Hibernating is {lo['customers_%']:.1f}% of customers and only {lo['revenue_%']:.1f}% of revenue, "
            f"so retention budget should not go here first.")
print(len(insights), "insights")

14 insights


## 11. Preprocessing for Clustering

**Business intent:** K-Means uses Euclidean distance, so a few wholesale whales would dominate every distance. We apply `log1p` (not `log`: Recency can be 0) and `StandardScaler`. We cluster the **raw R, F, M values** (not the 1-5 scores, which erase differences inside a score), never `CustomerID`.

In [49]:
# Preprocessing for clustering
# Business intent: K-Means uses Euclidean distance, so a few wholesale whales (Monetary up to ~279k)
# would dominate every distance. log1p compresses the right skew; StandardScaler puts R, F and M on the same scale.
X_raw = rfm[["Recency", "Frequency", "Monetary"]].copy()   # CustomerID stays as the index, never a feature
X_log = np.log1p(X_raw)                                     # log1p, not log: Recency can be 0

print("Skew before:", X_raw.skew().round(2).to_dict())
print("Skew after :", X_log.skew().round(2).to_dict())

scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X_log), index=X_raw.index, columns=X_raw.columns)
print("\nMean after scaling:", X.mean().round(3).to_dict())
print("Std after scaling :", X.std(ddof=0).round(3).to_dict())

fig = make_subplots(rows=2, cols=3, subplot_titles=[f"{c} (raw)" for c in X_raw] + [f"log1p({c})" for c in X_raw])
for i, col in enumerate(X_raw.columns, 1):
    fig.add_trace(go.Histogram(x=X_raw[col], nbinsx=50, marker_color="#8D99AE", showlegend=False), row=1, col=i)
    fig.add_trace(go.Histogram(x=X_log[col], nbinsx=50, marker_color="#08D9D6", showlegend=False), row=2, col=i)
fig.update_layout(title="RFM before and after the log transform", height=550)
fig.show()

log_decision("Clustering features", "Clustering on raw Recency, Frequency, Monetary -> log1p -> StandardScaler (not on 1-5 scores, not on CustomerID)",
             len(X), "Prevents wholesale whales from dominating distances; keeps within-score differences that quantile scores erase")

Skew before: {'Recency': 1.24, 'Frequency': 11.83, 'Monetary': 21.21}
Skew after : {'Recency': -0.47, 'Frequency': 1.22, 'Monetary': 0.36}

Mean after scaling: {'Recency': 0.0, 'Frequency': 0.0, 'Monetary': -0.0}
Std after scaling : {'Recency': 1.0, 'Frequency': 1.0, 'Monetary': 1.0}


**Interpretation.** The log transform cuts skew from 1.24 / 11.83 / 21.21 (R / F / M) to -0.47 / 1.22 / 0.36. Frequency is still right-skewed (many customers sit at exactly 1 order) and Recency is slightly over-corrected (-0.47); both are documented limitations. After scaling, every variable has mean 0 and standard deviation 1, so no variable dominates the Euclidean distance.

## 12. Clustering & Evaluation

**Business intent:** build defensible segments, not just a number of clusters. K is chosen with several criteria (elbow, silhouette, Davies-Bouldin, smallest cluster size, stability across subsamples), not by how the elbow *looks*.

### 12.1 Choosing K (all customers)

In [50]:
# Clustering: choose K with several criteria, not with the elbow alone
rows = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X)
    sizes = np.bincount(km.labels_)
    rows.append({
        "k": k,
        "wcss": km.inertia_,
        "silhouette": silhouette_score(X, km.labels_),     # 4,333 customers: full computation is fast
        "davies_bouldin": davies_bouldin_score(X, km.labels_),   # lower is better
        "smallest_cluster_%": sizes.min() / len(X) * 100,
    })
ev = pd.DataFrame(rows).set_index("k")
ev["wcss_drop_%"] = (-ev["wcss"].pct_change() * 100).round(1)
display(ev.round(3))

fig = make_subplots(rows=1, cols=2, subplot_titles=["Elbow: WCSS", "Silhouette (higher is better)"])
fig.add_trace(go.Scatter(x=ev.index, y=ev["wcss"], mode="lines+markers", line=dict(color="#FF2E63"), showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=ev.index, y=ev["silhouette"], mode="lines+markers", line=dict(color="#08D9D6"), showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="K", dtick=1)
fig.update_layout(title="Choosing K: Elbow and Silhouette", height=420)
fig.show()

,wcss,silhouette,davies_bouldin,smallest_cluster_%,wcss_drop_%
k,,,,,
2,6472.204,0.434,0.891,38.195,NaN
3,4852.719,0.340,1.043,16.986,25.0
4,3938.512,0.339,1.012,15.878,18.8
5,3307.801,0.316,0.992,6.577,16.0
6,2857.075,0.312,1.018,7.385,13.6
7,2568.903,0.293,1.015,5.239,10.1
8,2340.245,0.288,1.010,4.500,8.9
9,2158.913,0.278,1.012,4.662,7.7
10,1998.405,0.277,1.026,3.393,7.4


**Interpretation.** The elbow has **no sharp bend**: the WCSS drop per extra cluster shrinks gradually (25.0%, 18.8%, 16.0%, 13.6%, 10.1%...), so the elbow alone cannot pick K. Silhouette is highest at K=2 (0.434), but that only splits active from inactive customers, which is too coarse for different actions. From K=3 on it is flat around 0.31-0.34 (K=3: 0.340, K=4: 0.339, K=5: 0.316). A silhouette near 0.3 means the groups overlap: the segments are a practical grouping on a continuum, not natural islands. K therefore has to be chosen with stability and business usefulness as well.

### 12.2 Stability and cluster profiles

In [51]:
# Stability: fit on a random 80% of customers, assign everyone, compare with the full fit (ARI = 1 means identical)
def stability(k, n_rep=20):
    base = KMeans(k, n_init=10, random_state=SEED).fit(X).labels_
    scores = []
    for r in range(n_rep):
        idx = np.random.RandomState(r).choice(len(X), int(0.8 * len(X)), replace=False)
        m = KMeans(k, n_init=10, random_state=r).fit(X.iloc[idx])
        scores.append(adjusted_rand_score(base, m.predict(X)))
    return round(np.mean(scores), 3), round(np.min(scores), 3)

stab = pd.DataFrame({k: stability(k) for k in [2, 3, 4, 5, 6]}, index=["mean_ARI", "min_ARI"]).T
print("Stability across 20 subsamples:")
display(stab)

# Cluster profiles in ORIGINAL units (medians are robust to whales)
def profile(k):
    t = rfm[["Recency", "Frequency", "Monetary"]].copy()
    t["cluster"] = KMeans(k, n_init=10, random_state=SEED).fit_predict(X)
    g = t.groupby("cluster").agg(customers=("Recency", "size"), med_recency=("Recency", "median"),
                                 med_orders=("Frequency", "median"), med_spend=("Monetary", "median"),
                                 revenue=("Monetary", "sum"))
    g["customers_%"] = g["customers"] / g["customers"].sum() * 100
    g["revenue_%"] = g["revenue"] / g["revenue"].sum() * 100
    return g.sort_values("med_recency").round(1)

for k in [3, 4, 5]:
    print(f"\n=== K = {k} ===")
    display(profile(k))

Stability across 20 subsamples:


,mean_ARI,min_ARI
2,0.993,0.979
3,0.957,0.912
4,0.963,0.904
5,0.928,0.829
6,0.905,0.841



=== K = 3 ===


,customers,med_recency,med_orders,med_spend,revenue,customers_%,revenue_%
cluster,,,,,,,
0,736,8.0,10.0,3623.5,5571257.6,17.0,66.2
2,1687,31.0,3.0,996.1,2176067.4,38.9,25.9
1,1910,155.0,1.0,290.6,666877.5,44.1,7.9



=== K = 4 ===


,customers,med_recency,med_orders,med_spend,revenue,customers_%,revenue_%
cluster,,,,,,,
3,688,8.0,10.0,3715.4,5412407.3,15.9,64.3
0,817,16.0,2.0,468.4,449056.4,18.9,5.3
2,1187,53.0,4.0,1339.5,1996061.9,27.4,23.7
1,1641,173.0,1.0,292.5,556676.9,37.9,6.6



=== K = 5 ===


,customers,med_recency,med_orders,med_spend,revenue,customers_%,revenue_%
cluster,,,,,,,
3,285,4.0,17.0,6675.7,4152622.2,6.6,49.4
4,959,16.0,6.0,1961.5,2299888.8,22.1,27.3
1,815,21.0,2.0,364.0,339742.7,18.8,4.0
2,1051,81.0,3.0,939.8,1302818.8,24.3,15.5
0,1223,204.0,1.0,233.8,319129.9,28.2,3.8


**Interpretation.** Stability (mean ARI over 20 subsamples) falls as K grows: 0.993 (K=2), 0.957 (K=3), 0.963 (K=4), 0.928 (K=5), 0.905 (K=6). The profiles show what each K adds: K=3 separates 736 top customers (17.0%, 66.2% of revenue, median 10 orders), a middle group and 1,910 inactive customers (median 155 days, 7.9% of revenue). K=4 splits out recent low-spend buyers (817 customers, median 2 orders, 16 days). K=5 isolates a very small elite (285 customers, 6.6% of customers, **49.4% of revenue**, median 17 orders).

### 12.3 First K-Means result (K = 5, all customers)

In [52]:
# First K-Means result (K = 5); clusters ranked by median spend (1 = most valuable)
km5 = KMeans(5, n_init=10, random_state=SEED).fit(X)
t = rfm[["Recency", "Frequency", "Monetary", "Segment"]].copy()
t["raw"] = km5.labels_
rank = {c: i + 1 for i, c in enumerate(t.groupby("raw")["Monetary"].median().sort_values(ascending=False).index)}
t["Cluster"] = t["raw"].map(rank)
rfm["Cluster"] = t["Cluster"]

g = t.groupby("Cluster").agg(customers=("Recency", "size"), med_recency=("Recency", "median"),
                             med_orders=("Frequency", "median"), med_spend=("Monetary", "median"),
                             revenue=("Monetary", "sum"))
g["customers_%"] = (g["customers"] / g["customers"].sum() * 100).round(1)
g["revenue_%"] = (g["revenue"] / g["revenue"].sum() * 100).round(1)
display(g.round(1))

# Does the clustering add anything beyond the rule-based RFM segments? (row % of each cluster)
display(pd.crosstab(t["Cluster"], t["Segment"], normalize="index").mul(100).round(0))

smp = t.sample(2500, random_state=SEED)
fig = px.scatter_3d(smp, x="Recency", y="Frequency", z="Monetary", color=smp["Cluster"].astype(str),
                    log_y=True, log_z=True, opacity=0.7, title="K-Means clusters (K=5) in R-F-M space (log F and M)",
                    color_discrete_sequence=["#FF2E63", "#F3722C", "#F9C74F", "#08D9D6", "#8D99AE"])
fig.update_traces(marker_size=3)
fig.show()

log_decision("K exploration (all customers)", "K-Means K=5 on all 4,333 customers was explored first; it was later superseded (see Final segmentation)",
             len(rfm), "Without isolating the whales, one cluster mixed trade accounts with regular loyal buyers; the K=5 result was not stable once whales were set aside")

,customers,med_recency,med_orders,med_spend,revenue,customers_%,revenue_%
Cluster,,,,,,,
1,285,4.0,17.0,6675.7,4152622.2,6.6,49.4
2,959,16.0,6.0,1961.5,2299888.8,22.1,27.3
3,1051,81.0,3.0,939.8,1302818.8,24.3,15.5
4,815,21.0,2.0,364.0,339742.7,18.8,4.0
5,1223,204.0,1.0,233.8,319129.9,28.2,3.8


Segment,Champions,Loyal,New Inflow,About to Sleep,At-Risk High-Value,Lost / Hibernating
Cluster,,,,,,
1,97.0,2.0,0.0,0.0,1.0,0.0
2,52.0,45.0,1.0,0.0,2.0,0.0
3,0.0,25.0,1.0,17.0,28.0,28.0
4,0.0,16.0,60.0,24.0,0.0,0.0
5,0.0,0.0,0.0,14.0,0.0,86.0


**Interpretation.** In this first K=5 result, cluster 1 (285 customers) is 97% rule-based "Champions" and carries 49.4% of revenue, and cluster 5 (1,223 customers) is 86% "Lost / Hibernating". The clusters refine the rule-based segments: for example, the Champions are split into an elite and a regular group (52% of cluster 2 are Champions, 45% Loyal). The 3D plot shows clusters touching each other, in line with the low silhouette. Before accepting this K=5, we check whether the elite cluster is just a few extreme customers.

### 12.4 Hierarchical clustering and DBSCAN as checks
Hierarchical shows the tree structure and DBSCAN finds extreme customers (it is used as an **outlier detector**, not as the final segmentation).

In [53]:
# Hierarchical (Ward): dendrogram on a 500-customer sample (readable), Ward with 5 clusters on everyone
Xs = X.sample(500, random_state=SEED)
fig = ff.create_dendrogram(Xs.values, linkagefun=lambda a: linkage(a, "ward"))
fig.update_layout(title="Ward dendrogram (500 sampled customers): long vertical jumps suggest natural cut heights",
                  height=450, xaxis=dict(showticklabels=False))
fig.show()

hc = AgglomerativeClustering(n_clusters=5, linkage="ward").fit_predict(X)
print("Hierarchical (K=5): silhouette =", round(silhouette_score(X, hc), 3),
      "| ARI vs K-Means =", round(adjusted_rand_score(rfm["Cluster"], hc), 3))

# DBSCAN: used to find noise / extreme customers, not as the final segmentation
nn = NearestNeighbors(n_neighbors=10).fit(X)
kd = np.sort(nn.kneighbors(X)[0][:, -1])
fig = px.line(y=kd, title="Sorted 10-nearest-neighbour distance: the knee suggests eps",
              labels={"y": "distance", "index": "customers (sorted)"})
fig.show()

rows = []
for eps in [0.2, 0.3, 0.4, 0.5, 0.6, 0.8]:
    lab = DBSCAN(eps=eps, min_samples=10).fit_predict(X)
    rows.append({"eps": eps, "clusters": len(set(lab) - {-1}), "noise_%": round((lab == -1).mean() * 100, 1)})
display(pd.DataFrame(rows))

Hierarchical (K=5): silhouette = 0.272 | ARI vs K-Means = 0.517


,eps,clusters,noise_%
0,0.2,15,29.1
1,0.3,6,10.5
2,0.4,3,4.4
3,0.5,2,2.5
4,0.6,1,1.2
5,0.8,1,0.7


**Interpretation.**
- **Dendrogram:** the tree has a few large merges at the top, so a small number of groups is reasonable, but there is no single obvious cut height.
- **Hierarchical (Ward, K=5):** silhouette 0.272 (below K-Means 0.316) and ARI 0.517 against K-Means: the two methods agree only partly, which confirms that boundaries between segments are not sharp.
- **DBSCAN:** the k-distance curve is flat for most customers (95th percentile 0.50) and rises sharply in the tail (99th percentile 0.88, max 2.31); we use eps = 0.6, inside that knee. At eps 0.6 DBSCAN returns **one dense cluster plus 1.2% noise**, and at small eps it fragments into 15 clusters with 29% noise. So DBSCAN is not a useful segmenter here, but it is a good **outlier detector** for the extreme customers.

In [54]:
# DBSCAN as an outlier detector (eps = 0.6 from the k-distance knee, min_samples = 10)
db = DBSCAN(eps=0.6, min_samples=10).fit_predict(X)
rfm["Outlier"] = db == -1
print("Outliers:", rfm["Outlier"].sum(), f"({rfm['Outlier'].mean():.1%})")

display(rfm.groupby("Outlier")[["Recency", "Frequency", "Monetary"]].median())
out = rfm[rfm["Outlier"]]
print(f"Outliers' share of total revenue: {out['Monetary'].sum() / rfm['Monetary'].sum():.1%}")
display(pd.crosstab(rfm["Outlier"], rfm["Cluster"]))
display(out.sort_values("Monetary", ascending=False).head(10)[["Recency", "Frequency", "Monetary", "Segment", "Cluster"]])

# Is the Key Accounts cluster driven by a handful of customers?
k1 = rfm.loc[rfm["Cluster"] == 1, "Monetary"].sort_values(ascending=False)
print(f"\nCluster 1: top 10 customers = {k1.head(10).sum() / k1.sum():.1%} of the cluster's revenue; median spend {k1.median():,.0f}")

Outliers: 52 (1.2%)


,Recency,Frequency,Monetary
Outlier,,,
False,51.0,2.0,653.38
True,7.5,26.5,31436.68


Outliers' share of total revenue: 28.1%


Cluster,1,2,3,4,5
Outlier,,,,,
False,250,956,1047,810,1218
True,35,3,4,5,5


,Recency,Frequency,Monetary,Segment,Cluster
CustomerID,,,,,
14646,1,72,279138.02,Champions,1
18102,0,60,259657.30,Champions,1
17450,8,46,194390.79,Champions,1
14911,1,196,131810.93,Champions,1
12415,24,20,124564.53,Champions,1
14156,9,54,116560.08,Champions,1
17511,2,30,86766.38,Champions,1
16029,38,59,69220.89,Loyal,1
16684,4,28,66653.56,Champions,1



Cluster 1: top 10 customers = 33.6% of the cluster's revenue; median spend 6,676


**Interpretation.** DBSCAN flags 52 customers (1.2%). Their median is 26.5 orders and 31,437 spend (vs 2 orders and 653 for the rest), they ordered recently (median 7.5 days), and together they hold **28.1% of identified spend**. 35 of the 52 fall into K=5 cluster 1, whose top 10 customers alone are 33.6% of the cluster's revenue. So the "elite cluster" is partly a group of extreme accounts, not a typical segment.

### 12.5 Whales: outliers set aside, then K re-chosen for everyone else

In [55]:
# Who are the outliers that are NOT in Cluster 1?
rest = rfm[rfm["Outlier"] & (rfm["Cluster"] != 1)]
display(rest[["Recency", "Frequency", "Monetary", "Segment", "Cluster"]].sort_values("Monetary", ascending=False))

# Re-check K after setting the 52 outliers aside (K-Means is sensitive to extreme points)
Xn = X[~rfm["Outlier"]]
rows = []
for k in range(3, 8):
    lab = KMeans(k, n_init=10, random_state=SEED).fit_predict(Xn)
    rows.append({"k": k, "silhouette": round(silhouette_score(Xn, lab), 3),
                 "davies_bouldin": round(davies_bouldin_score(Xn, lab), 3),
                 "smallest_cluster_%": round(np.bincount(lab).min() / len(Xn) * 100, 1)})
display(pd.DataFrame(rows).set_index("k"))

,Recency,Frequency,Monetary,Segment,Cluster
CustomerID,,,,,
15749,235,3,44534.30,At-Risk High-Value,3
12753,22,6,21429.39,Champions,2
16000,2,3,12393.70,Loyal,2
12409,78,2,11067.72,Lost / Hibernating,3
12590,211,1,9341.26,Lost / Hibernating,3
13093,275,8,7832.47,At-Risk High-Value,3
15195,2,1,3861.00,New Inflow,2
12713,0,1,794.55,New Inflow,4
17988,11,7,191.17,Loyal,4


,silhouette,davies_bouldin,smallest_cluster_%
k,,,
3,0.331,1.074,20.3
4,0.337,1.008,17.5
5,0.317,1.017,12.8
6,0.313,0.995,9.8
7,0.310,0.977,6.8


**Interpretation.** The 17 outliers outside cluster 1 are a mix: genuine large spenders (e.g. customer 15749: 44,534, silent for 235 days; 12753: 21,429) and odd low-spend combinations (e.g. 16446: 2.90). After setting all 52 outliers aside, K=4 has the best silhouette (0.337, vs 0.331 for K=3 and 0.317 for K=5) with the smallest cluster at 17.5% of customers.

In [56]:
# Whale rule: DBSCAN outlier AND very high spend (some outliers are odd low-spend combinations)
out = rfm[rfm["Outlier"]]
for pq in [.90, .95, .99]:
    thr = rfm["Monetary"].quantile(pq)
    w = out[out["Monetary"] >= thr]
    print(f"Outlier & Monetary >= P{int(pq * 100)} ({thr:,.0f}): {len(w)} customers | {w['Monetary'].sum() / rfm['Monetary'].sum():.1%} of identified spend")

print("\nOutliers below the P90 spend (odd cases, not whales):")
display(out[out["Monetary"] < rfm["Monetary"].quantile(.90)][["Recency", "Frequency", "Monetary", "Segment", "Cluster"]])

Outlier & Monetary >= P90 (3,540): 41 customers | 28.0% of identified spend
Outlier & Monetary >= P95 (5,711): 39 customers | 27.9% of identified spend
Outlier & Monetary >= P99 (17,235): 31 customers | 26.9% of identified spend

Outliers below the P90 spend (odd cases, not whales):


,Recency,Frequency,Monetary,Segment,Cluster
CustomerID,,,,,
12713,0,1,794.55,New Inflow,4
14792,63,1,6.20,About to Sleep,5
14865,7,2,52.20,New Inflow,4
15992,3,1,41.99,New Inflow,4
16446,205,1,2.90,Lost / Hibernating,5
16454,64,1,5.90,About to Sleep,5
16738,297,1,3.75,Lost / Hibernating,5
17816,36,4,36.56,Loyal,4
17961,21,39,1296.44,Champions,1


**Interpretation.** A DBSCAN outlier is not automatically a whale: 11 of the 52 outliers spend less than the 90th percentile (some only a few pounds). So the rule is **outlier AND spend >= 95th percentile (5,711)**, which gives 39 customers and 27.9% of identified spend. Using the 90th, 95th or 99th percentile gives 41, 39 or 31 customers and 28.0%, 27.9% or 26.9% of spend: the share of spend is stable even when the count changes.

In [57]:
# Final whale rule (adjustable): DBSCAN outlier AND spend >= P95 of all identified customers
WHALE_P = 0.95
thr = rfm["Monetary"].quantile(WHALE_P)
rfm["Whale"] = rfm["Outlier"] & (rfm["Monetary"] >= thr)
print(f"Whales: {rfm['Whale'].sum()} customers ({rfm['Whale'].mean():.1%}) | "
      f"{rfm.loc[rfm['Whale'], 'Monetary'].sum() / rfm['Monetary'].sum():.1%} of identified spend | threshold {thr:,.0f}")

# Choose K for everyone else: silhouette + stability (subsample ARI) + smallest cluster
Xr = X[~rfm["Whale"]]

def stability_r(k, n_rep=20):
    base = KMeans(k, n_init=10, random_state=SEED).fit(Xr).labels_
    s = []
    for r in range(n_rep):
        idx = np.random.RandomState(r).choice(len(Xr), int(0.8 * len(Xr)), replace=False)
        m = KMeans(k, n_init=10, random_state=r).fit(Xr.iloc[idx])
        s.append(adjusted_rand_score(base, m.predict(Xr)))
    return round(np.mean(s), 3), round(np.min(s), 3)

ev2 = []
for k in [3, 4, 5]:
    lab = KMeans(k, n_init=10, random_state=SEED).fit_predict(Xr)
    ari = stability_r(k)
    ev2.append({"k": k, "silhouette": round(silhouette_score(Xr, lab), 3), "mean_ARI": ari[0], "min_ARI": ari[1],
                "smallest_cluster_%": round(np.bincount(lab).min() / len(Xr) * 100, 1)})
display(pd.DataFrame(ev2).set_index("k"))

# Profiles of K=4 and K=5 without whales (original units, medians); revenue_% is out of ALL identified spend
for k in [4, 5]:
    t2 = rfm.loc[~rfm["Whale"], ["Recency", "Frequency", "Monetary"]].copy()
    t2["c"] = KMeans(k, n_init=10, random_state=SEED).fit_predict(Xr)
    g2 = t2.groupby("c").agg(customers=("Recency", "size"), med_recency=("Recency", "median"),
                             med_orders=("Frequency", "median"), med_spend=("Monetary", "median"), revenue=("Monetary", "sum"))
    g2["customers_%"] = (g2["customers"] / g2["customers"].sum() * 100).round(1)
    g2["revenue_%"] = (g2["revenue"] / rfm["Monetary"].sum() * 100).round(1)
    print(f"\n=== K = {k} (without whales) ===")
    display(g2.sort_values("med_recency").round(1))

Whales: 39 customers (0.9%) | 27.9% of identified spend | threshold 5,711


,silhouette,mean_ARI,min_ARI,smallest_cluster_%
k,,,,
3,0.329,0.981,0.909,20.3
4,0.337,0.928,0.868,17.3
5,0.316,0.785,0.415,13.1



=== K = 4 (without whales) ===


,customers,med_recency,med_orders,med_spend,revenue,customers_%,revenue_%
c,,,,,,,
3,743,8.0,9.0,3190.6,3300954.6,17.3,39.2
2,826,18.0,2.0,412.8,397789.1,19.2,4.7
0,1183,56.0,4.0,1269.8,1852788.8,27.6,22.0
1,1542,184.0,1.0,289.6,512857.8,35.9,6.1



=== K = 5 (without whales) ===


,customers,med_recency,med_orders,med_spend,revenue,customers_%,revenue_%
c,,,,,,,
1,564,10.0,11.0,4008.1,3011836.2,13.1,35.8
3,659,10.0,4.0,1123.9,869682.4,15.3,10.3
4,904,35.0,1.0,314.9,306667.5,21.1,3.6
2,1059,71.0,3.0,1142.9,1547186.1,24.7,18.4
0,1108,232.0,1.0,256.1,329018.2,25.8,3.9


**Interpretation.** With the 39 whales set aside, K=5 is no longer stable (min ARI 0.415; the mean ARI is 0.785), while K=4 keeps the best silhouette (0.337), a mean ARI of 0.928 and every cluster at or above 17.3% of customers. K=3 is more stable (mean ARI 0.981) and has a similar silhouette (0.329). K=4 is chosen because it separates groups that need different actions (see 13.1), at the cost of some stability. The K=4 profile: 743 customers who ordered ~8 days ago with 9 orders (core), 826 recent customers with 2 orders, 1,183 customers last seen after 56 days with 4 orders, and 1,542 customers inactive for 184 days with 1 order.

## 13. Segment Profiling

**Business intent:** final segmentation = **Whales** (DBSCAN outlier *and* spend at or above the 95th percentile) + **K-Means K=4** on everyone else. Names are given *after* looking at the statistics, and renamed if the data does not support them.

In [58]:
# Final segmentation: Whales + K-Means on everyone else
# Why 4 and not 3? K=3 profile (original units, no whales)
t3 = rfm.loc[~rfm["Whale"], ["Recency", "Frequency", "Monetary"]].copy()
t3["c"] = KMeans(3, n_init=10, random_state=SEED).fit_predict(Xr)
g3 = t3.groupby("c").agg(customers=("Recency", "size"), med_recency=("Recency", "median"),
                         med_orders=("Frequency", "median"), med_spend=("Monetary", "median"), revenue=("Monetary", "sum"))
g3["customers_%"] = (g3["customers"] / g3["customers"].sum() * 100).round(1)
g3["revenue_%"] = (g3["revenue"] / rfm["Monetary"].sum() * 100).round(1)
print("=== K = 3 (without whales) ===")
display(g3.sort_values("med_recency").round(1))

# Final labels: K=4 on non-whales, clusters ranked by median spend, then named from their statistics
FINAL_K = 4
tmp = rfm.loc[~rfm["Whale"], ["Monetary"]].copy()
tmp["raw"] = KMeans(FINAL_K, n_init=10, random_state=SEED).fit_predict(Xr)
rank = {c: i + 1 for i, c in enumerate(tmp.groupby("raw")["Monetary"].median().sort_values(ascending=False).index)}
NAMES = {1: "Core Loyal", 2: "Fading Regulars", 3: "Recent Light Buyers", 4: "Dormant One-Timers"}
rfm["FinalSegment"] = "Whales"
rfm.loc[~rfm["Whale"], "FinalSegment"] = tmp["raw"].map(rank).map(NAMES)

# Profile: every name must be backed by these numbers
order = ["Whales", "Core Loyal", "Fading Regulars", "Recent Light Buyers", "Dormant One-Timers"]
final = rfm.join(cust[["avg_order_value", "return_ratio"]]).groupby("FinalSegment").agg(
    customers=("Recency", "size"), med_recency=("Recency", "median"), med_orders=("Frequency", "median"),
    med_spend=("Monetary", "median"), revenue=("Monetary", "sum"),
    med_order_value=("avg_order_value", "median"), avg_return_ratio=("return_ratio", "mean")).reindex(order)
final["customers_%"] = final["customers"] / final["customers"].sum() * 100
final["revenue_%"] = final["revenue"] / final["revenue"].sum() * 100
final["revenue_per_customer"] = final["revenue"] / final["customers"]
display(final.round(2))

sp = final.reset_index()[["FinalSegment", "customers_%", "revenue_%"]].melt("FinalSegment", var_name="metric", value_name="percent")
fig = px.bar(sp, x="FinalSegment", y="percent", color="metric", barmode="group",
             color_discrete_sequence=["#08D9D6", "#FF2E63"], title="Final segments: share of customers vs share of identified spend")
fig.show()

log_decision("Final segmentation", "Whales (DBSCAN outlier AND spend >= P95, 39 customers) + K-Means K=4 on the rest, named from segment statistics",
             len(rfm), "K=5 became unstable without the whales (min ARI 0.415); K=4 has the best silhouette (0.337) and every segment >= 17% of customers")

=== K = 3 (without whales) ===


,customers,med_recency,med_orders,med_spend,revenue,customers_%,revenue_%
c,,,,,,,
1,871,10.0,8.0,3014.3,3671244.5,20.3,43.6
2,1652,32.0,3.0,872.3,1800785.7,38.5,21.4
0,1771,169.0,1.0,276.1,592360.2,41.2,7.0


,customers,med_recency,med_orders,med_spend,revenue,med_order_value,avg_return_ratio,customers_%,revenue_%,revenue_per_customer
FinalSegment,,,,,,,,,,
Whales,39,4.0,46.0,42055.96,2349812.06,1770.69,0.04,0.90,27.93,60251.59
Core Loyal,743,8.0,9.0,3190.55,3300954.64,347.28,0.02,17.15,39.23,4442.74
Fading Regulars,1183,56.0,4.0,1269.84,1852788.83,354.25,0.02,27.30,22.02,1566.18
Recent Light Buyers,826,18.0,2.0,412.80,397789.12,220.94,0.01,19.06,4.73,481.58
Dormant One-Timers,1542,184.0,1.0,289.57,512857.80,219.81,0.02,35.59,6.10,332.59


**Interpretation.** **Why K=4 and not K=3?** K=3 (without whales) gives a single middle group of 1,652 customers (median 32 days, 3 orders) between the top and dormant groups, so it cannot tell recent light buyers (18 days, 2 orders) from regulars (56 days, 4 orders). Those two need different actions (second-purchase offer vs reorder reminder), so K=4 is more useful, even though K=3 is a bit more stable.

**Final segments (the table still shows the first name "Fading Regulars", which is revised in 13.3):**
- **Whales:** 39 customers (0.9%), 27.9% of spend, median 46 orders, 60,252 spend per customer.
- **Core Loyal:** 743 customers (17.2%), 39.2% of spend, median 9 orders, last order 8 days ago.
- **Regulars (first called "Fading"):** 1,183 customers (27.3%), 22.0% of spend.
- **Recent Light Buyers:** 826 customers (19.1%), 4.7% of spend.
- **Dormant One-Timers:** 1,542 customers (35.6%), 6.1% of spend.

Whales and Core Loyal together are 18% of customers and 67.2% of spend.

In [59]:
# Does the final segmentation add anything beyond the rule-based RFM segments? (row % of each final segment)
display(pd.crosstab(rfm["FinalSegment"], rfm["Segment"], normalize="index").mul(100).round(0).reindex(order))

# Is "Fading Regulars" really overdue? Compare current recency with the customers' own usual gap between orders
chk = rfm.join(cust[["median_gap_days"]])
gap = chk.groupby("FinalSegment").agg(med_recency=("Recency", "median"), med_gap_days=("median_gap_days", "median"),
                                       customers_with_gap=("median_gap_days", "count")).reindex(order)
gap["recency_vs_gap"] = (gap["med_recency"] / gap["med_gap_days"]).round(1)
display(gap)

# Sensitivity of the whale rule to the DBSCAN eps (we chose 0.6 from the k-distance knee)
rows = []
for eps in [0.5, 0.6, 0.7]:
    o = DBSCAN(eps=eps, min_samples=10).fit_predict(X) == -1
    w = o & (rfm["Monetary"].values >= thr)
    rows.append({"eps": eps, "outliers": int(o.sum()), "whales": int(w.sum()),
                 "whale_spend_%": round(rfm.loc[w, "Monetary"].sum() / rfm["Monetary"].sum() * 100, 1)})
display(pd.DataFrame(rows).set_index("eps"))

Segment,Champions,Loyal,New Inflow,About to Sleep,At-Risk High-Value,Lost / Hibernating
FinalSegment,,,,,,
Whales,85.0,5.0,0.0,0.0,5.0,5.0
Core Loyal,87.0,13.0,0.0,0.0,1.0,0.0
Fading Regulars,9.0,46.0,2.0,9.0,26.0,9.0
Recent Light Buyers,0.0,23.0,60.0,17.0,0.0,0.0
Dormant One-Timers,0.0,0.0,0.0,19.0,1.0,80.0


,med_recency,med_gap_days,customers_with_gap,recency_vs_gap
FinalSegment,,,,
Whales,4.0,5.25,38,0.8
Core Loyal,8.0,26.50,743,0.3
Fading Regulars,56.0,58.50,1143,1.0
Recent Light Buyers,18.0,78.50,514,0.2
Dormant One-Timers,184.0,67.00,387,2.7


,outliers,whales,whale_spend_%
eps,,,
0.5,108,61,31.9
0.6,52,39,27.9
0.7,41,31,25.8


**Interpretation.**
- **Beyond the rule-based segments:** 85% of Whales and 87% of Core Loyal customers are both "Champions" in the rule-based view, although their median spend differs 13x (42,056 vs 3,191). The clustering adds this split; the rules cannot.
- **Is "Fading" justified?** The regulars' median recency is 56 days and their usual gap between orders is 58.5 days (ratio 1.0): they are at their usual repurchase point, not fading. By contrast, Dormant One-Timers are at 2.7x their usual gap. The name is revised in the next cell.
- **Whale rule sensitivity:** eps 0.5 / 0.6 / 0.7 give 61 / 39 / 31 whales holding 31.9% / 27.9% / 25.8% of spend. The count depends on eps; the finding "about 1% of customers hold 26-32% of spend" does not.

In [60]:
# Rename: "Fading" is not supported (recency 56 vs usual gap 58.5, ratio 1.0): they are at their usual repurchase point
rfm["FinalSegment"] = rfm["FinalSegment"].replace({"Fading Regulars": "Due-Now Regulars"})
order = ["Whales", "Core Loyal", "Due-Now Regulars", "Recent Light Buyers", "Dormant One-Timers"]
final = final.rename(index={"Fading Regulars": "Due-Now Regulars"}).reindex(order)

# Per-customer check: is a customer past THEIR OWN usual gap? (repeat buyers only: one-time buyers have no gap)
chk = rfm.join(cust[["median_gap_days"]])
chk = chk[chk["median_gap_days"].notna()].copy()
chk["overdue"] = chk["Recency"] > chk["median_gap_days"]
chk["very_overdue"] = chk["Recency"] > 2 * chk["median_gap_days"]
od = chk.groupby("FinalSegment").agg(repeat_buyers=("overdue", "size"), overdue=("overdue", "mean"),
                                      very_overdue=("very_overdue", "mean")).reindex(order)
od[["overdue", "very_overdue"]] = (od[["overdue", "very_overdue"]] * 100).round(0)
display(od.rename(columns={"overdue": "overdue_%", "very_overdue": "very_overdue_%"}))

log_decision("Segment naming", "'Fading Regulars' renamed 'Due-Now Regulars': their median recency (56 d) equals their usual order gap (58.5 d)",
             int((rfm["FinalSegment"] == "Due-Now Regulars").sum()), "Avoids a misleading 'fading' label; the action is a timely reorder reminder, not a heavy win-back offer")

,repeat_buyers,overdue_%,very_overdue_%
FinalSegment,,,
Whales,38,34.0,29.0
Core Loyal,743,12.0,4.0
Due-Now Regulars,1143,45.0,24.0
Recent Light Buyers,514,12.0,7.0
Dormant One-Timers,387,75.0,59.0


**Interpretation.** "Fading Regulars" is renamed **"Due-Now Regulars"**, because the numbers do not support "fading". Looking at each customer against their own usual gap: 45% of Due-Now Regulars' repeat buyers are already past it (24% beyond twice the gap), compared with 12% of Core Loyal and Recent Light Buyers and 75% of Dormant One-Timers (59% beyond twice the gap). For whales the ratio is misleading because their usual gap is only ~5 days; section 14 reports their silence in real days.

In [61]:
f = final
add_insight(f"Whales: {f.loc['Whales','customers']:.0f} customers ({f.loc['Whales','customers_%']:.1f}%) generate {f.loc['Whales','revenue_%']:.1f}% of identified spend "
            f"(median {f.loc['Whales','med_orders']:.0f} orders, recency {f.loc['Whales','med_recency']:.0f} days). They behave like trade accounts, but the data cannot confirm it. "
            f"The count depends on the DBSCAN eps (31 to 61 customers, 26-32% of spend).")

top2 = f.loc[["Whales", "Core Loyal"]]
add_insight(f"Concentration: Whales + Core Loyal are {top2['customers_%'].sum():.1f}% of customers but {top2['revenue_%'].sum():.1f}% of identified spend, "
            f"while Recent Light Buyers + Dormant One-Timers are {f.loc[['Recent Light Buyers','Dormant One-Timers'],'customers_%'].sum():.1f}% of customers and only "
            f"{f.loc[['Recent Light Buyers','Dormant One-Timers'],'revenue_%'].sum():.1f}% of spend.")

whale_quiet = rfm[(rfm["FinalSegment"] == "Whales") & rfm["Segment"].isin(["At-Risk High-Value", "Lost / Hibernating"])]
add_insight(f"Rule-based RFM puts Whales and Core Loyal together in 'Champions', but median spend differs {f.loc['Whales','med_spend'] / f.loc['Core Loyal','med_spend']:.0f}x "
            f"({f.loc['Whales','med_spend']:,.0f} vs {f.loc['Core Loyal','med_spend']:,.0f}). {len(whale_quiet)} whales are rule-labelled At-Risk or Lost "
            f"(spend {whale_quiet['Monetary'].sum():,.0f}): the highest-value win-back targets.")

dn = od.loc["Due-Now Regulars"]
add_insight(f"Due-Now Regulars ({f.loc['Due-Now Regulars','customers']:.0f} customers, {f.loc['Due-Now Regulars','revenue_%']:.1f}% of spend) have median recency "
            f"{f.loc['Due-Now Regulars','med_recency']:.0f} days, about their usual gap between orders; {dn['overdue']:.0f}% of the repeat buyers are already past their own gap. "
            f"A timely reorder reminder is a hypothesis worth testing before they lapse.")

add_insight(f"Dormant One-Timers are {f.loc['Dormant One-Timers','customers_%']:.1f}% of customers but {f.loc['Dormant One-Timers','revenue_%']:.1f}% of spend, "
            f"inactive for a median {f.loc['Dormant One-Timers','med_recency']:.0f} days. Heavy discounting here is hard to justify on value alone.")
w = rfm[rfm["FinalSegment"] == "Whales"]
add_insight(f"Whale silence in real days: {int((w['Recency'] > 30).sum())} of {len(w)} whales have not ordered for more than 30 days "
            f"(longest: {w['Recency'].max():.0f} days), while the median whale ordered {w['Recency'].median():.0f} days ago. "
            f"Their usual gap is only ~5 days, so a ratio to the gap would overstate the problem.")
print(len(insights), "insights")


matrix = pd.DataFrame({
    "segment": order,
    "evidence": [f"{f.loc[s,'customers']:.0f} customers, {f.loc[s,'revenue_%']:.1f}% of spend, recency {f.loc[s,'med_recency']:.0f} d, {f.loc[s,'med_orders']:.0f} orders" for s in order],
    "strategy": ["Dedicated account management, early access, review of terms; win back the quiet ones first",
                 "Loyalty perks and cross-sell of related product ranges",
                 "Reorder reminder before their usual gap ends; bundle offer",
                 "Welcome / second-purchase sequence with a small incentive",
                 "Low-cost reactivation email only; no heavy discount"],
    "kpi_to_test": ["Retention of the whales; spend per whale", "Orders per customer; share moving up",
                    "Share reordering within their usual gap", "Second-purchase rate within 60 days", "Reactivation rate vs campaign cost"],
})
pd.set_option("display.max_colwidth", 200)
matrix

20 insights


,segment,evidence,strategy,kpi_to_test
0,Whales,"39 customers, 27.9% of spend, recency 4 d, 46 orders","Dedicated account management, early access, review of terms; win back the quiet ones first",Retention of the whales; spend per whale
1,Core Loyal,"743 customers, 39.2% of spend, recency 8 d, 9 orders",Loyalty perks and cross-sell of related product ranges,Orders per customer; share moving up
2,Due-Now Regulars,"1183 customers, 22.0% of spend, recency 56 d, 4 orders",Reorder reminder before their usual gap ends; bundle offer,Share reordering within their usual gap
3,Recent Light Buyers,"826 customers, 4.7% of spend, recency 18 d, 2 orders",Welcome / second-purchase sequence with a small incentive,Second-purchase rate within 60 days
4,Dormant One-Timers,"1542 customers, 6.1% of spend, recency 184 d, 1 orders",Low-cost reactivation email only; no heavy discount,Reactivation rate vs campaign cost


## 14. Business Recommendations

All actions below are **hypotheses to test** (e.g. with a control group). The data is descriptive: it shows who behaves how, not what would change their behaviour.

| Segment | Evidence | Recommended action | KPI to test |
|---|---|---|---|
| **Whales** | 39 customers (0.9%), 27.9% of identified spend, median 46 orders, last order 4 days ago. 5 of 39 silent for 30+ days (longest 275) | Dedicated account management and early access. Contact the silent ones first (4 are rule-labelled At-Risk/Lost, spend 72,776) | Whales retained; spend per whale |
| **Core Loyal** | 743 customers (17.2%), 39.2% of spend, median 9 orders, last order 8 days ago | Loyalty perks; cross-sell related product ranges | Orders per customer; share moving up |
| **Due-Now Regulars** | 1,183 customers (27.3%), 22.0% of spend. Median recency 56 days vs usual gap 58.5; 45% of repeat buyers already past their own gap | Reorder reminder before the usual gap ends; bundle offer | Share reordering within their usual gap |
| **Recent Light Buyers** | 826 customers (19.1%), 4.7% of spend, median 2 orders, last order 18 days ago | Welcome / second-purchase sequence with a small incentive | Second-purchase rate within 60 days |
| **Dormant One-Timers** | 1,542 customers (35.6%), 6.1% of spend, 1 order, inactive for a median 184 days | Low-cost reactivation email only; no heavy discount on value alone | Reactivation rate vs campaign cost |

**Priority order:** (1) protect the whales, because losing a few accounts moves revenue the most; (2) act on Due-Now Regulars while they are still within reach; (3) the rest.

**Operational recommendations from the EDA**
- **Concentration risk:** the top 20% of identified customers generate 74% of identified revenue, and 39 whales alone 27.9%. Revenue depends on a small group.
- **Timing:** revenue peaks on weekdays between 10:00 and 15:00 and no transactions are recorded on Saturdays. Plan maintenance outside these windows and test ad timing around them.
- **Seasonality:** Sep-Nov 2011 is about 36% of the period's revenue (Nov 1.43M, ~2.3x the Jan-Aug average). With one year of data this is an observation, not a confirmed yearly pattern. Plan Q4 stock early and validate with a second year.
- **Returns:** two mistaken orders (~75-81k units each) were cancelled 12-16 minutes after being placed and make up 51.7% of returned value; without them returns are 2.3% of gross sales. Validating unusually large quantities at order time is a hypothesis worth testing.
- **Products:** 21.5% of products generate 80% of revenue. Dead stock is small (2.4% of high-volume products, 0.7% of historical revenue), so it is not a clearance priority.
- **Guest checkout:** about 15% of net revenue comes from invoices without a CustomerID. Encouraging account creation would bring this revenue into customer analysis.

### Evidence behind the recommendations
The full insights register (all numbers are computed in the code above) and the business matrix exported for the dashboard.

In [62]:
for i, t in enumerate(insights, 1):
    print(f"{i}. {t}\n")
matrix

1. Products: only 5 of the top-10 products by units are also in the top-10 by revenue. Popcorn Holder leads in units (~56k) but ranks 10 by revenue; Regency Cakestand 3 Tier leads revenue (~164k) but ranks 44 by units. Price per unit, not just volume, decides what drives revenue.

2. Time: revenue from Sep-Nov 2011 is ~3.5M, about 36% of the whole period, and Nov 2011 (1.43M) is ~2.3x the Jan-Aug monthly average (~634k). Descriptive only: with ~1 year of data we cannot call it a recurring seasonal pattern, and Dec 2011 is incomplete (ends on the 9th).

3. Customers: the top 20% of identifiable customers generate 74% of identified revenue, and 30.4% of customers bought only once. Revenue is concentrated, and a large share of customers never came back in the observed period.

4. Hours/days: revenue peaks in the hours 10:00-15:59 (top hours: 12, 10, 15, 13, 11) and no transactions are recorded on Saturday.

5. Countries: the UK accounts for 3,942 of the 4,362 identified customers and 84.8

,segment,evidence,strategy,kpi_to_test
0,Whales,"39 customers, 27.9% of spend, recency 4 d, 46 orders","Dedicated account management, early access, review of terms; win back the quiet ones first",Retention of the whales; spend per whale
1,Core Loyal,"743 customers, 39.2% of spend, recency 8 d, 9 orders",Loyalty perks and cross-sell of related product ranges,Orders per customer; share moving up
2,Due-Now Regulars,"1183 customers, 22.0% of spend, recency 56 d, 4 orders",Reorder reminder before their usual gap ends; bundle offer,Share reordering within their usual gap
3,Recent Light Buyers,"826 customers, 4.7% of spend, recency 18 d, 2 orders",Welcome / second-purchase sequence with a small incentive,Second-purchase rate within 60 days
4,Dormant One-Timers,"1542 customers, 6.1% of spend, recency 184 d, 1 orders",Low-cost reactivation email only; no heavy discount,Reactivation rate vs campaign cost


## 15. Limitations & Next Steps

**Limitations**
1. **One year of data** (1 Dec 2010 to 9 Dec 2011, last month incomplete). Seasonality cannot be confirmed; no Saturday transactions are recorded.
2. **24.9% of rows (3,710 invoices, about 15% of net revenue) have no CustomerID.** Segments describe the 4,333 identifiable customers only.
3. **No cost or margin data.** Revenue is not profit, and refund value is not the cost of returns.
4. **Monetary = sum of positive purchases;** returns are tracked separately. Orders cancelled within 60 minutes with the same customer, product and quantity are treated as mistaken and removed from customer spend (a heuristic rule). 29 customers with only cancellation lines were left out.
5. **Segments overlap.** The silhouette is about 0.34 (K=4), so segments are a practical grouping on a continuum, not natural islands. Customers near a boundary could belong to the neighbouring segment. Hierarchical clustering agrees only partly with K-Means (ARI 0.517).
6. **The whale definition is sensitive.** DBSCAN eps between 0.5 and 0.7 gives 31 to 61 whales (26-32% of spend). We report 39 (eps = 0.6, spend at or above the 95th percentile). The stable finding: about 1% of customers hold roughly 26-32% of identified spend.
7. **Modelling choices matter:** log1p slightly over-corrects Recency (skew -0.47); Frequency uses fixed bins instead of qcut; removing 5,268 duplicate rows assumes they are recording errors; 14.7% of cancellation lines have no original purchase in the data.
8. **Snapshot in time:** segments are as of 10 Dec 2011. "Wholesale" behaviour of the whales is plausible but not confirmed by the data.

**Next steps**
- A/B test each action against a control group before scaling.
- Add a second year of data, or validate by time: build segments on the early months and check how customers behave afterwards.
- Add product-mix features (what each segment buys) and margin data; move from revenue to profit.
- Build churn and customer-lifetime-value models on top of the segments.
- Find ways to identify guest-checkout customers.

## 16. Final Conclusion

The business depends on a small group of customers. **39 whales (0.9%)** generate 27.9% of identified spend, and together with **Core Loyal** customers (743) they are 18% of customers but 67.2% of spend. At the other end, **Recent Light Buyers and Dormant One-Timers** are 54.7% of customers but only 10.8% of spend.

Rule-based RFM puts whales and Core Loyal customers in the same "Champions" group even though their median spend differs 13x (42,056 vs 3,191); isolating the outliers with DBSCAN and clustering the rest with K-Means (K=4, stable across subsamples: mean ARI 0.93) gave five segments that each call for a different action. Data quality shaped the result: a quarter of rows had no CustomerID, and two mistaken orders made up half of all returned value.

The recommended focus is to protect the whales, remind the Due-Now Regulars before they lapse, and avoid heavy discounts on dormant one-time buyers. These are hypotheses backed by descriptive evidence; each should be tested with a control group before scaling.

## Appendix: Export processed tables

**Business intent:** save every table the Streamlit app and the PDF report need, so they read the same numbers as this notebook. Output folder: `data/processed/`.

In [63]:
# Save the processed tables used by the Streamlit app and the PDF report
OUT.mkdir(parents=True, exist_ok=True)

seg = (rfm.rename(columns={"Cluster": "Cluster_K5_all"})
          .join(cust[["avg_order_value", "avg_units_per_order", "median_gap_days", "return_ratio", "country"]]))
seg.index.name = "CustomerID"
seg.reset_index().to_csv(OUT / "customers_segmented.csv", index=False)
final.reset_index().to_csv(OUT / "segment_profile.csv", index=False)
matrix.to_csv(OUT / "business_matrix.csv", index=False)
pd.DataFrame(decision_log).to_csv(OUT / "decision_log.csv", index=False)
pd.DataFrame({"insight": insights}).to_csv(OUT / "insights.csv", index=False)
monthly.to_csv(OUT / "monthly_revenue.csv", index=False)
country.reset_index().to_csv(OUT / "country_summary.csv", index=False)
prod.sort_values("revenue", ascending=False).head(300).reset_index().to_csv(OUT / "top_products.csv", index=False)
buys_clean.pivot_table(index="DayOfWeek", columns="Hour", values="TotalLineRevenue", aggfunc="sum").reset_index().to_csv(OUT / "weekday_hour.csv", index=False)

# Headline KPIs for the dashboard (one row, computed here so the app never recomputes them)
kpis = pd.DataFrame([{
    "gross_sales": gross_sold, "returned_value": returned_total, "net_revenue": sales["TotalLineRevenue"].sum(),
    "return_rate_pct": returned_total / gross_sold * 100,
    "orders": int(sales.loc[~sales["IsCancellation"], "InvoiceNo"].nunique()),
    "identified_customers": int(len(seg)), "countries": int(sales["Country"].nunique()),
    "uk_share_pct": float(country.loc["United Kingdom", "revenue_share_%"]),
    "first_date": str(sales["InvoiceDate"].min().date()), "last_date": str(sales["InvoiceDate"].max().date()),
}])
kpis.to_csv(OUT / "kpis.csv", index=False)

print("Saved:", sorted(p.name for p in OUT.iterdir()), "| customers:", len(seg))
display(kpis.T)

Saved: ['business_matrix.csv', 'country_summary.csv', 'customers_segmented.csv', 'decision_log.csv', 'insights.csv', 'kpis.csv', 'monthly_revenue.csv', 'segment_profile.csv', 'top_products.csv', 'weekday_hour.csv'] | customers: 4333


,0
gross_sales,10247907.683
returned_value,475901.16
net_revenue,9772006.523
return_rate_pct,4.643886
orders,19773
identified_customers,4333
countries,38
uk_share_pct,84.8
first_date,2010-12-01
last_date,2011-12-09
